# CAP Agentic AI — V4b (Kaggle)
## V4 + Three-Layer Guideline-Aware Extraction

### What's new in V4b vs V4

| Component | V4 | V4b |
|-----------|----|----|
| **Feature extraction** | Generic field list, guideline-blind | Three-layer: Python pre-processing → guideline-aware LLM → Python flag annotation |
| **Renal function** | LLM extracts raw value | Python calculates CrCl (Cockcroft-Gault) and eGFR (CKD-EPI 2021) from creatinine if needed; classifies into guideline renal categories |
| **Effective DOT** | LLM estimates day number | Python calculates cumulative effective days across all antibiotics (e.g. ceftriaxone 4d + amoxicillin day 5 = DOT 9) |
| **CORB / SMART-COP** | LLM extracts if documented | Python scores from component observations at onset |
| **IV-to-oral switch** | LLM estimates | Python evaluates all 7 criteria individually |
| **Dosing check** | LLM reasoning | Python lookup against parsed Tables 1–8 for drug + renal category |
| **Spectrum check** | LLM reasoning | Python lookup against guideline inappropriate prescribing indicator lists |
| **Guideline flags** | None | Python-generated GUIDELINE_FLAG annotations appended to feature summary before decision calls |
| **All V4 memory layers** | ✅ | ✅ Fully preserved |

### Memory layers preserved from V4
ShortTermMemory · LongTermMemory · SemanticMemory · EpisodicMemory ·
AbstractMemory · ErrorMemory · PolicyMemory · ObservationalMemory ·
ExperienceMemory · CaseSummaryMemory


In [ ]:
# ============================================================================
# CELL 1: ENVIRONMENT SETUP
# ============================================================================
import os, subprocess, sys

print("Installing dependencies...")
for pkg in ["python-docx", "sentence-transformers", "accelerate"]:
    result = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                            capture_output=True, text=True)
    print(f"  {'✅' if result.returncode==0 else '❌'} {pkg}")

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
print(f"\n  Python  : {sys.version.split()[0]}")
print(f"  PyTorch : {torch.__version__}")
if torch.cuda.is_available():
    print(f"  GPU     : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("  ⚠️  No GPU — enable in Settings → Accelerator → GPU T4")


In [ ]:
# ============================================================================
# CELL 2: KAGGLE PATH CONFIGURATION
# ============================================================================
import os, glob

class KagglePaths:
    INPUT_BASE     = "/kaggle/input"
    WORKING        = "/kaggle/working"
    CHECKPOINT_DIR = f"{WORKING}/checkpoints"
    OUTPUT_DIR     = f"{WORKING}/outputs"
    EVAL_DIR       = f"{WORKING}/evaluation"

    @classmethod
    def find_file(cls, filename):
        matches = glob.glob(f"{cls.INPUT_BASE}/**/{filename}", recursive=True)
        return matches[0] if matches else None

    @classmethod
    def verify_files(cls):
        print("Verifying Kaggle input files...")
        ok = True
        for label, fname in [("cycle_A_B.jsonl", "cycle_A_B.jsonl"),
                              ("CAP Guideline (.docx)", None)]:
            if fname:
                path = cls.find_file(fname)
            else:
                hits = glob.glob(f"{cls.INPUT_BASE}/**/*.docx", recursive=True)
                path = hits[0] if hits else None
            if path:
                print(f"  ✅ {label} ({os.path.getsize(path)/1024:.0f} KB) → {path}")
            else:
                print(f"  ❌ {label} NOT FOUND"); ok = False
        for d in [cls.CHECKPOINT_DIR, cls.OUTPUT_DIR, cls.EVAL_DIR]:
            os.makedirs(d, exist_ok=True)
        print(f"  ✅ Output dirs ready under {cls.WORKING}")
        return ok

paths = KagglePaths()
DATA_FILE      = paths.find_file("cycle_A_B.jsonl")
GUIDELINE_FILE = paths.find_file("CAP_Guideline_v2.docx")
paths.verify_files()
print(f"\nDATA_FILE      = {DATA_FILE}")
print(f"GUIDELINE_FILE = {GUIDELINE_FILE}")


In [ ]:
# ============================================================================
# CELL 3: CHECKPOINT RESUME CONFIGURATION
# ============================================================================
RESUME_FROM = None   # e.g. "/kaggle/input/cap-checkpoints/latest_checkpoint.pkl"
print(f"Resume mode : {'ON  → ' + RESUME_FROM if RESUME_FROM else 'OFF (fresh start)'}")


In [ ]:

# ============================================================================
# CELL 4: V4b CORE
# ============================================================================
# Three-layer extraction architecture:
#   Layer 1 — Python pre-processing (renal, DOT, CORB, SMART-COP, IV-oral switch)
#   Layer 2 — Guideline-aware LLM extraction call
#   Layer 3 — Python annotation (dosing lookup, spectrum check, indicator flags)
#
# All V4 memory systems fully preserved and unchanged:
#   ShortTermMemory, LongTermMemory, SemanticMemory, EpisodicMemory,
#   AbstractMemory, ErrorMemory, PolicyMemory, ObservationalMemory,
#   ExperienceMemory, CaseSummaryMemory
# ============================================================================

import json, os, gc, pickle, re, warnings, math
from pathlib import Path
from datetime import datetime
from collections import defaultdict, deque
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix
import pandas as pd
from tqdm import tqdm
from docx import Document

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

from sentence_transformers import SentenceTransformer
_EMBEDDER = None
def get_embedder():
    global _EMBEDDER
    if _EMBEDDER is None:
        _EMBEDDER = SentenceTransformer('all-MiniLM-L6-v2')
    return _EMBEDDER


# =============================================================================
# SECTION 1: CLINICAL CONSTANTS FROM GUIDELINE
# All drug lists, dosing tables, and indicator lists are derived directly
# from the CAP guideline document.
# =============================================================================

# ── Drug name normalisation map ───────────────────────────────────────────────
# Maps brand names / abbreviations → canonical names matching dosing table keys
DRUG_ALIASES = {
    # Beta-lactams
    'augmentin': 'amoxicillin-clavulanate (iv)',
    'amoxyclav': 'amoxicillin-clavulanate (iv)',
    'tazocin': 'piperacillin-tazobactam',
    'pip-tazo': 'piperacillin-tazobactam',
    'piptazo': 'piperacillin-tazobactam',
    'pip/tazo': 'piperacillin-tazobactam',
    'benzylpen': 'benzylpenicillin',
    'benzpen': 'benzylpenicillin',
    'penicillin g': 'benzylpenicillin',
    'fluclox': 'flucloxacillin (iv)',
    'dicloxacillin': 'flucloxacillin (iv)',
    'ceftriax': 'ceftriaxone',
    'rocephin': 'ceftriaxone',
    'cefurox': 'cefuroxime (po)',
    'zinnat': 'cefuroxime (po)',
    'merrem': 'meropenem',
    'invanz': 'ertapenem',
    # Macrolides
    'zithromax': 'azithromycin (po)',
    'azithro': 'azithromycin (po)',
    'klacid': 'clarithromycin (po)',
    'biaxin': 'clarithromycin (po)',
    # Fluoroquinolones
    'avelox': 'moxifloxacin (po)',
    'tavanic': 'levofloxacin (po)',
    'cipro': 'ciprofloxacin (po)',
    # Glycopeptides
    'vanc': 'vancomycin (iv)',
    'vanco': 'vancomycin (iv)',
    'targocid': 'teicoplanin (iv)',
    # Other
    'doxy': 'doxycycline (po)',
    'tamiflu': 'oseltamivir (po)',
    'bactrim': 'trimethoprim-sulfamethoxazole (po/iv)',
    'septrin': 'trimethoprim-sulfamethoxazole (po/iv)',
    'flagyl': 'metronidazole (po/iv)',
    'dalacin': 'clindamycin (po/iv)',
}

# ── Dosing table — parsed from Tables 1-8 of guideline ───────────────────────
# Format: (canonical_drug_name_lower, renal_category) → recommended_dose_string
# Renal categories: Normal, Mild, Moderate, Severe, IHD, CRRT, PD
DOSING_TABLE: Dict[Tuple[str, str], str] = {
    # Table 1 / Table 3 — Beta-lactams and monobactams
    ('ampicillin (iv)',              'Normal'):   '2g q4-6h',
    ('ampicillin (iv)',              'Mild'):     '2g q4-6h',
    ('ampicillin (iv)',              'Moderate'): '2g q8h',
    ('ampicillin (iv)',              'Severe'):   '1-2g q12h',
    ('ampicillin (iv)',              'IHD'):      '1-2g q12h + post-HD',
    ('ampicillin (iv)',              'CRRT'):     '2g q6-8h',
    ('ampicillin (iv)',              'PD'):       '1g q12h',
    ('amoxicillin (iv)',             'Normal'):   '2g q6h',
    ('amoxicillin (iv)',             'Mild'):     '2g q6h',
    ('amoxicillin (iv)',             'Moderate'): '2g q8-12h',
    ('amoxicillin (iv)',             'Severe'):   '2g q12h',
    ('amoxicillin (iv)',             'IHD'):      '2g q12h post-HD',
    ('amoxicillin (iv)',             'CRRT'):     '2g q6-8h',
    ('amoxicillin (iv)',             'PD'):       '2g q12h',
    ('amoxicillin (po)',             'Normal'):   '500mg q8h or 1g q8h',
    ('amoxicillin (po)',             'Mild'):     '500mg q8h or 1g q8h',
    ('amoxicillin (po)',             'Moderate'): '500mg-1g q12h',
    ('amoxicillin (po)',             'Severe'):   '500mg q12-24h',
    ('amoxicillin (po)',             'IHD'):      '500mg q24h post-HD',
    ('amoxicillin (po)',             'CRRT'):     '500mg-1g q8-12h',
    ('amoxicillin (po)',             'PD'):       '500mg q24h',
    ('amoxicillin-clavulanate (iv)', 'Normal'):   '1.2g q6h',
    ('amoxicillin-clavulanate (iv)', 'Mild'):     '1.2g q6h',
    ('amoxicillin-clavulanate (iv)', 'Moderate'): '1.2g q8-12h',
    ('amoxicillin-clavulanate (iv)', 'Severe'):   '1.2g q12h',
    ('amoxicillin-clavulanate (iv)', 'IHD'):      '1.2g q12h post-HD',
    ('amoxicillin-clavulanate (iv)', 'CRRT'):     '1.2g q6-8h',
    ('amoxicillin-clavulanate (iv)', 'PD'):       '1.2g q12-24h',
    ('aztreonam',                    'Normal'):   '2g q8h',
    ('aztreonam',                    'Mild'):     '2g q8h',
    ('aztreonam',                    'Moderate'): '1-2g q12h',
    ('aztreonam',                    'Severe'):   '1g q24h',
    ('aztreonam',                    'IHD'):      '1g post-HD',
    ('aztreonam',                    'CRRT'):     '2g q8-12h',
    ('aztreonam',                    'PD'):       '1g q24h',
    ('benzylpenicillin',             'Normal'):   'Normal dose per indication',
    ('benzylpenicillin',             'Mild'):     '75% dose at usual interval',
    ('benzylpenicillin',             'Moderate'): '50% dose at usual interval',
    ('benzylpenicillin',             'Severe'):   '25% dose at usual interval',
    ('benzylpenicillin',             'IHD'):      '25% dose at usual interval',
    ('benzylpenicillin',             'CRRT'):     '50% dose at usual interval',
    ('benzylpenicillin',             'PD'):       '25% dose at usual interval',
    ('flucloxacillin (iv)',          'Normal'):   '2g q6h',
    ('flucloxacillin (iv)',          'Mild'):     '2g q6h',
    ('flucloxacillin (iv)',          'Moderate'): '2g q6h',
    ('flucloxacillin (iv)',          'Severe'):   '1g q6-8h',
    ('flucloxacillin (iv)',          'IHD'):      '1g q6-8h',
    ('flucloxacillin (iv)',          'CRRT'):     '2g q6-8h',
    ('flucloxacillin (iv)',          'PD'):       '1g q6-8h',
    ('flucloxacillin (po)',          'Normal'):   '500mg q6h',
    ('flucloxacillin (po)',          'Mild'):     '500mg q6h',
    ('flucloxacillin (po)',          'Moderate'): '500mg q6h',
    ('flucloxacillin (po)',          'Severe'):   '500mg q6h',
    ('flucloxacillin (po)',          'IHD'):      '500mg q6h',
    ('flucloxacillin (po)',          'CRRT'):     '500mg q6h',
    ('flucloxacillin (po)',          'PD'):       '500mg q6h',
    ('piperacillin-tazobactam',      'Normal'):   '4.5g q6h',
    ('piperacillin-tazobactam',      'Mild'):     '4.5g q6h',
    ('piperacillin-tazobactam',      'Moderate'): '4.5g q8h',
    ('piperacillin-tazobactam',      'Severe'):   '4.5g q12h',
    ('piperacillin-tazobactam',      'IHD'):      '4.5g post-HD',
    ('piperacillin-tazobactam',      'CRRT'):     '4.5g q6-8h',
    ('piperacillin-tazobactam',      'PD'):       '4.5g q12h',
    ('cefazolin',                    'Normal'):   '2g q8h',
    ('cefazolin',                    'Mild'):     '2g q8h',
    ('cefazolin',                    'Moderate'): '2g q12h',
    ('cefazolin',                    'Severe'):   '1g q24h',
    ('cefazolin',                    'IHD'):      '2-3g post-HD',
    ('cefazolin',                    'CRRT'):     '2g q8h',
    ('cefazolin',                    'PD'):       '1g q24h',
    ('ceftriaxone',                  'Normal'):   '2g q24h',
    ('ceftriaxone',                  'Mild'):     '2g q24h',
    ('ceftriaxone',                  'Moderate'): '2g q24h',
    ('ceftriaxone',                  'Severe'):   '2g q24h',
    ('ceftriaxone',                  'IHD'):      '2g q24h',
    ('ceftriaxone',                  'CRRT'):     '2g q24h',
    ('ceftriaxone',                  'PD'):       '2g q24h',
    ('ceftazidime',                  'Normal'):   '2g q8h',
    ('ceftazidime',                  'Mild'):     '2g q8h',
    ('ceftazidime',                  'Moderate'): '2g q12h',
    ('ceftazidime',                  'Severe'):   '1g q24h',
    ('ceftazidime',                  'IHD'):      '1g post-HD',
    ('ceftazidime',                  'CRRT'):     '2g q8h',
    ('ceftazidime',                  'PD'):       '1g q24h',
    ('cefepime',                     'Normal'):   '2g q8h',
    ('cefepime',                     'Mild'):     '2g q8-12h',
    ('cefepime',                     'Moderate'): '2g q12h',
    ('cefepime',                     'Severe'):   '1g q12h',
    ('cefepime',                     'IHD'):      '1g post-HD',
    ('cefepime',                     'CRRT'):     '2g q12h',
    ('cefepime',                     'PD'):       '1g q24h',
    ('cefuroxime (po)',               'Normal'):   '500mg q12h',
    ('cefuroxime (po)',               'Mild'):     '500mg q12h',
    ('cefuroxime (po)',               'Moderate'): '250-500mg q12h',
    ('cefuroxime (po)',               'Severe'):   '250mg q24h',
    ('cefuroxime (po)',               'IHD'):      '250mg q24h post-HD',
    ('cefuroxime (po)',               'CRRT'):     '500mg q12h',
    ('cefuroxime (po)',               'PD'):       '250mg q24h',
    ('meropenem',                    'Normal'):   '1g q8h',
    ('meropenem',                    'Mild'):     '1g q8h',
    ('meropenem',                    'Moderate'): '1g q12h',
    ('meropenem',                    'Severe'):   '500mg q12h',
    ('meropenem',                    'IHD'):      '500mg post-HD',
    ('meropenem',                    'CRRT'):     '1g q8h',
    ('meropenem',                    'PD'):       '500mg q24h',
    ('imipenem',                     'Normal'):   '500mg q6h',
    ('imipenem',                     'Mild'):     '500mg q6h',
    ('imipenem',                     'Moderate'): '500mg q8h',
    ('imipenem',                     'Severe'):   '250mg q12h',
    ('imipenem',                     'IHD'):      '250mg post-HD',
    ('imipenem',                     'CRRT'):     '500mg q8-12h',
    ('imipenem',                     'PD'):       '250mg q12-24h',
    ('ertapenem',                    'Normal'):   '1g q24h',
    ('ertapenem',                    'Mild'):     '1g q24h',
    ('ertapenem',                    'Moderate'): '500mg q24h',
    ('ertapenem',                    'Severe'):   '500mg q24h',
    ('ertapenem',                    'IHD'):      '500mg post-HD',
    ('ertapenem',                    'CRRT'):     'AVOID',
    ('ertapenem',                    'PD'):       '500mg q24h',
    # Table 4 — Fluoroquinolones
    ('ciprofloxacin (iv)',            'Normal'):   '400mg q8-12h',
    ('ciprofloxacin (iv)',            'Mild'):     '400mg q12h',
    ('ciprofloxacin (iv)',            'Moderate'): '400mg q24h',
    ('ciprofloxacin (iv)',            'Severe'):   '200-400mg q24h',
    ('ciprofloxacin (iv)',            'IHD'):      '200-400mg post-HD',
    ('ciprofloxacin (iv)',            'CRRT'):     '400mg q12h',
    ('ciprofloxacin (iv)',            'PD'):       '200-400mg q24h',
    ('ciprofloxacin (po)',            'Normal'):   '500-750mg q12h',
    ('ciprofloxacin (po)',            'Mild'):     '500-750mg q12h',
    ('ciprofloxacin (po)',            'Moderate'): '500mg q24h',
    ('ciprofloxacin (po)',            'Severe'):   '250-500mg q24h',
    ('ciprofloxacin (po)',            'IHD'):      '250-500mg q24h post-HD',
    ('ciprofloxacin (po)',            'CRRT'):     '500-750mg q12h',
    ('ciprofloxacin (po)',            'PD'):       '250-500mg q24h',
    ('levofloxacin (iv)',             'Normal'):   '750mg q24h',
    ('levofloxacin (iv)',             'Mild'):     '750mg q24h',
    ('levofloxacin (iv)',             'Moderate'): '750mg q48h or 500mg q24h',
    ('levofloxacin (iv)',             'Severe'):   '500mg q48h',
    ('levofloxacin (iv)',             'IHD'):      '500mg post-HD then q48h',
    ('levofloxacin (iv)',             'CRRT'):     '750mg q24h',
    ('levofloxacin (iv)',             'PD'):       '500mg q48h',
    ('levofloxacin (po)',             'Normal'):   '750mg q24h',
    ('levofloxacin (po)',             'Mild'):     '750mg q24h',
    ('levofloxacin (po)',             'Moderate'): '750mg q48h or 500mg q24h',
    ('levofloxacin (po)',             'Severe'):   '500mg q48h',
    ('levofloxacin (po)',             'IHD'):      '500mg post-HD then q48h',
    ('levofloxacin (po)',             'CRRT'):     '750mg q24h',
    ('levofloxacin (po)',             'PD'):       '500mg q48h',
    ('moxifloxacin (iv)',             'Normal'):   '400mg q24h',
    ('moxifloxacin (iv)',             'Mild'):     '400mg q24h',
    ('moxifloxacin (iv)',             'Moderate'): '400mg q24h',
    ('moxifloxacin (iv)',             'Severe'):   '400mg q24h',
    ('moxifloxacin (iv)',             'IHD'):      '400mg q24h',
    ('moxifloxacin (iv)',             'CRRT'):     '400mg q24h',
    ('moxifloxacin (iv)',             'PD'):       '400mg q24h',
    ('moxifloxacin (po)',             'Normal'):   '400mg q24h',
    ('moxifloxacin (po)',             'Mild'):     '400mg q24h',
    ('moxifloxacin (po)',             'Moderate'): '400mg q24h',
    ('moxifloxacin (po)',             'Severe'):   '400mg q24h',
    ('moxifloxacin (po)',             'IHD'):      '400mg q24h',
    ('moxifloxacin (po)',             'CRRT'):     '400mg q24h',
    ('moxifloxacin (po)',             'PD'):       '400mg q24h',
    # Table 5 — Glycopeptides
    ('vancomycin (iv)',               'Normal'):   'Load 25mg/kg ABW; Maint 15mg/kg q12h',
    ('vancomycin (iv)',               'Mild'):     '15mg/kg q12h',
    ('vancomycin (iv)',               'Moderate'): '7.5mg/kg q12h',
    ('vancomycin (iv)',               'Severe'):   '7.5mg/kg STAT then TDM-guided',
    ('vancomycin (iv)',               'IHD'):      '15-20mg/kg post-HD (TDM-guided)',
    ('vancomycin (iv)',               'CRRT'):     '15-20mg/kg q12-24h',
    ('vancomycin (iv)',               'PD'):       '15mg/kg q48-72h',
    # Table 7 — Macrolides
    ('azithromycin (iv)',             'Normal'):   '500mg q24h',
    ('azithromycin (iv)',             'Mild'):     '500mg q24h',
    ('azithromycin (iv)',             'Moderate'): '500mg q24h',
    ('azithromycin (iv)',             'Severe'):   '500mg q24h',
    ('azithromycin (iv)',             'IHD'):      '500mg q24h',
    ('azithromycin (iv)',             'CRRT'):     '500mg q24h',
    ('azithromycin (iv)',             'PD'):       '500mg q24h',
    ('azithromycin (po)',             'Normal'):   '500mg q24h',
    ('azithromycin (po)',             'Mild'):     '500mg q24h',
    ('azithromycin (po)',             'Moderate'): '500mg q24h',
    ('azithromycin (po)',             'Severe'):   '500mg q24h',
    ('azithromycin (po)',             'IHD'):      '500mg q24h',
    ('azithromycin (po)',             'CRRT'):     '500mg q24h',
    ('azithromycin (po)',             'PD'):       '500mg q24h',
    ('clarithromycin (po)',           'Normal'):   '500mg q12h',
    ('clarithromycin (po)',           'Mild'):     '500mg q12h',
    ('clarithromycin (po)',           'Moderate'): '250mg q12h',
    ('clarithromycin (po)',           'Severe'):   '250mg q24h',
    ('clarithromycin (po)',           'IHD'):      '250mg q24h post-HD',
    ('clarithromycin (po)',           'CRRT'):     '250-500mg q12-24h',
    ('clarithromycin (po)',           'PD'):       '250mg q24h',
    ('clarithromycin (iv)',           'Normal'):   '500mg q12h',
    ('clarithromycin (iv)',           'Mild'):     '500mg q12h',
    ('clarithromycin (iv)',           'Moderate'): '250mg q12h',
    ('clarithromycin (iv)',           'Severe'):   '250mg q24h',
    ('clarithromycin (iv)',           'IHD'):      '250mg q24h post-HD',
    ('clarithromycin (iv)',           'CRRT'):     '250-500mg q12-24h',
    ('clarithromycin (iv)',           'PD'):       '250mg q24h',
    # Table 8 — Other antibiotics
    ('doxycycline (po)',              'Normal'):   '100mg q12h',
    ('doxycycline (po)',              'Mild'):     '100mg q12h',
    ('doxycycline (po)',              'Moderate'): '100mg q12h',
    ('doxycycline (po)',              'Severe'):   '100mg q12h',
    ('doxycycline (po)',              'IHD'):      '100mg q12h',
    ('doxycycline (po)',              'CRRT'):     '100mg q12h',
    ('doxycycline (po)',              'PD'):       '100mg q12h',
    ('metronidazole (po/iv)',         'Normal'):   '500mg q8-12h',
    ('metronidazole (po/iv)',         'Mild'):     '500mg q8-12h',
    ('metronidazole (po/iv)',         'Moderate'): '500mg q12h',
    ('metronidazole (po/iv)',         'Severe'):   '500mg q12h',
    ('metronidazole (po/iv)',         'IHD'):      '500mg q12h',
    ('metronidazole (po/iv)',         'CRRT'):     '500mg q8-12h',
    ('metronidazole (po/iv)',         'PD'):       '500mg q12h',
    ('clindamycin (po/iv)',           'Normal'):   '450mg PO q6-8h or 600-900mg IV q8h',
    ('clindamycin (po/iv)',           'Mild'):     '450mg PO q6-8h or 600-900mg IV q8h',
    ('clindamycin (po/iv)',           'Moderate'): '450mg PO q6-8h or 600-900mg IV q8h',
    ('clindamycin (po/iv)',           'Severe'):   '450mg PO q6-8h or 600-900mg IV q8h',
    ('clindamycin (po/iv)',           'IHD'):      '450mg PO q6-8h or 600-900mg IV q8h',
    ('clindamycin (po/iv)',           'CRRT'):     '450mg PO q6-8h or 600-900mg IV q8h',
    ('clindamycin (po/iv)',           'PD'):       '450mg PO q6-8h or 600-900mg IV q8h',
    ('linezolid (po/iv)',             'Normal'):   '600mg q12h',
    ('linezolid (po/iv)',             'Mild'):     '600mg q12h',
    ('linezolid (po/iv)',             'Moderate'): '600mg q12h',
    ('linezolid (po/iv)',             'Severe'):   '600mg q12h',
    ('linezolid (po/iv)',             'IHD'):      '600mg q12h',
    ('linezolid (po/iv)',             'CRRT'):     '600mg q12h',
    ('linezolid (po/iv)',             'PD'):       '600mg q12h',
    ('trimethoprim-sulfamethoxazole (po/iv)', 'Normal'):   '8-10mg/kg/day TMP q6-8h',
    ('trimethoprim-sulfamethoxazole (po/iv)', 'Mild'):     'Same',
    ('trimethoprim-sulfamethoxazole (po/iv)', 'Moderate'): '50% dose or q12h',
    ('trimethoprim-sulfamethoxazole (po/iv)', 'Severe'):   '25-50% dose or q24h',
    ('trimethoprim-sulfamethoxazole (po/iv)', 'IHD'):      '50% dose post-HD',
    ('trimethoprim-sulfamethoxazole (po/iv)', 'CRRT'):     '8-10mg/kg/day TMP',
    ('trimethoprim-sulfamethoxazole (po/iv)', 'PD'):       '25-50% dose',
    # Table 10 — Antivirals
    ('oseltamivir (po)',              'Normal'):   '75mg q12h',
    ('oseltamivir (po)',              'Mild'):     '75mg q12h',
    ('oseltamivir (po)',              'Moderate'): '75mg q24h',
    ('oseltamivir (po)',              'Severe'):   '30mg q24h',
    ('oseltamivir (po)',              'IHD'):      '30mg post-HD',
    ('oseltamivir (po)',              'CRRT'):     '75mg q12h',
    ('oseltamivir (po)',              'PD'):       '30mg q24h',
}

# Drugs with NO renal adjustment required (guideline-confirmed)
NO_RENAL_ADJUSTMENT = {
    'doxycycline (po)', 'azithromycin (iv)', 'azithromycin (po)',
    'moxifloxacin (iv)', 'moxifloxacin (po)', 'ceftriaxone',
    'flucloxacillin (po)', 'clindamycin (po/iv)', 'linezolid (po/iv)',
    'roxithromycin (po)', 'erythromycin (iv)', 'erythromycin (po)',
    'rifampicin (po/iv)', 'tigecycline (iv)', 'metronidazole (po/iv)',
    'sodium fusidate (po)',
}

# ── Spectrum indicator lists (from guideline, by severity) ───────────────────
# Drugs considered inappropriately BROAD for each CAP severity level
TOO_BROAD = {
    'mild': {
        'amoxicillin-clavulanate', 'ceftriaxone', 'piperacillin-tazobactam',
        'ceftazidime', 'cefepime', 'aztreonam', 'meropenem', 'ertapenem',
        'imipenem', 'ciprofloxacin', 'moxifloxacin', 'norfloxacin',
        'azithromycin', 'vancomycin', 'teicoplanin', 'linezolid',
        'tigecycline', 'metronidazole', 'clindamycin', 'lincomycin',
    },
    'moderate': {
        'amoxicillin-clavulanate', 'piperacillin-tazobactam',
        'ceftazidime', 'cefepime', 'aztreonam', 'meropenem', 'ertapenem',
        'imipenem', 'ciprofloxacin', 'moxifloxacin', 'norfloxacin',
        'azithromycin', 'vancomycin', 'teicoplanin', 'linezolid',
        'tigecycline', 'metronidazole', 'clindamycin', 'lincomycin',
    },
    'severe': {
        'amoxicillin-clavulanate', 'piperacillin-tazobactam',
        'ceftazidime', 'cefepime', 'aztreonam', 'meropenem', 'ertapenem',
        'imipenem', 'ciprofloxacin', 'norfloxacin',
        'vancomycin', 'teicoplanin', 'linezolid',
        'tigecycline', 'metronidazole', 'clindamycin', 'lincomycin',
    },
}

# Drugs considered TOO NARROW for each CAP severity level
TOO_NARROW = {
    'mild': {
        'cefazolin', 'cefalexin', 'cefoxitin', 'gentamicin', 'tobramycin',
        'amikacin', 'metronidazole', 'clindamycin', 'flucloxacillin',
        'nitrofurantoin', 'trimethoprim', 'trimethoprim-sulfamethoxazole',
        'daptomycin',
    },
    'moderate': {
        'cefazolin', 'cefalexin', 'cefoxitin', 'gentamicin', 'tobramycin',
        'amikacin', 'metronidazole', 'clindamycin', 'flucloxacillin',
        'nitrofurantoin', 'trimethoprim', 'trimethoprim-sulfamethoxazole',
        'daptomycin',
    },
    'severe': {
        'benzylpenicillin', 'doxycycline', 'clarithromycin',
        'cefazolin', 'cefalexin', 'cefoxitin', 'gentamicin', 'tobramycin',
        'amikacin', 'metronidazole', 'clindamycin', 'flucloxacillin',
        'nitrofurantoin', 'trimethoprim', 'trimethoprim-sulfamethoxazole',
        'daptomycin',
    },
}

# Duration thresholds (effective DOT) beyond which therapy is excessive (guideline)
DURATION_THRESHOLD = {
    'mild':      5,
    'moderate':  7,
    'severe':    7,
    'legionella': 7,
}

# Penicillin allergy cross-reactivity (guideline indicator P98/P185)
# Maps: allergy_type → set of drugs that are contraindicated
PEN_ALLERGY_CONTRAINDICATED = {
    'rash':          {'amoxicillin', 'benzylpenicillin', 'amoxicillin-clavulanate',
                      'flucloxacillin', 'piperacillin-tazobactam'},
    'severe':        {'amoxicillin', 'benzylpenicillin', 'amoxicillin-clavulanate',
                      'flucloxacillin', 'piperacillin-tazobactam',
                      'ceftriaxone', 'cefuroxime', 'cefazolin', 'cefalexin',
                      'ceftazidime', 'cefepime', 'meropenem', 'ertapenem',
                      'imipenem', 'aztreonam'},
    'anaphylaxis':   {'amoxicillin', 'benzylpenicillin', 'amoxicillin-clavulanate',
                      'flucloxacillin', 'piperacillin-tazobactam',
                      'ceftriaxone', 'cefuroxime', 'cefazolin', 'cefalexin',
                      'ceftazidime', 'cefepime', 'meropenem', 'ertapenem',
                      'imipenem', 'aztreonam'},
}


# =============================================================================
# SECTION 2: DATA STRUCTURES (unchanged from V4)
# =============================================================================

@dataclass
class CaseMemory:
    case_id:             str
    case_text:           str
    feature_summary:     str
    predictions:         Dict[str, str]
    ground_truth:        Dict[str, str]
    confidences:         Dict[str, int]
    reasoning:           Dict[str, str]
    reflexion_notes:     Dict[str, str]
    timestamp:           str
    performance_metrics: Dict[str, float]
    embedding:           Optional[np.ndarray] = field(default=None, repr=False)

@dataclass
class AbstractPrinciple:
    principle_id:     str
    marker:           str
    description:      str
    confidence:       float
    supporting_cases: List[str]
    created_at:       str
    last_updated:     str

@dataclass
class PolicyRule:
    rule_id:      str
    condition:    str
    action:       str
    priority:     int
    success_rate: float
    usage_count:  int

@dataclass
class ExperienceRule:
    rule_id:               str
    marker:                str
    clinical_rule:         str
    trigger_features:      List[str]
    mistake_type:          str
    source_case_id:        str
    embedding:             np.ndarray = field(default=None, repr=False)
    confidence:            float = 0.50
    times_retrieved:       int   = 0
    times_prevented_error: int   = 0
    times_contradicted:    int   = 0
    created_at:            str   = ""

@dataclass
class CaseSummaryEntry:
    summary_id:              str
    marker:                  str
    prediction:              str
    case_pattern:            str
    why_prediction:          str
    distinguishing_features: List[str]
    source_case_id:          str
    embedding:               np.ndarray = field(default=None, repr=False)
    confidence:              float = 4.0
    times_retrieved:         int   = 0
    times_confirmed:         int   = 0
    created_at:              str   = ""


# =============================================================================
# SECTION 3: MEMORY SYSTEMS (all unchanged from V4)
# =============================================================================

class ShortTermMemory:
    def __init__(self, capacity=5):
        self.memory = deque(maxlen=capacity)
    def add(self, case): self.memory.append(case)
    def get_recent(self, n=3): return list(self.memory)[-n:]

class LongTermMemory:
    def __init__(self):
        self.cases = []; self.embeddings = None
    def add(self, case):
        self.cases.append(case)
        emb = case.embedding
        if emb is not None:
            self.embeddings = emb.reshape(1,-1) if self.embeddings is None \
                              else np.vstack([self.embeddings, emb.reshape(1,-1)])
    def get_similar(self, qe, top_k=3, marker=None, prefer_yes=False):
        if self.embeddings is None or len(self.cases) < 3: return []
        q = qe / (np.linalg.norm(qe)+1e-9)
        M = self.embeddings / (np.linalg.norm(self.embeddings,axis=1,keepdims=True)+1e-9)
        sims = M @ q
        if prefer_yes and marker:
            for i,c in enumerate(self.cases):
                if c.ground_truth.get(marker)=='YES': sims[i]*=1.3
        return [self.cases[i] for i in np.argsort(sims)[::-1][:top_k]]

class SemanticMemory:
    MARKER_SECTION = {
        'antimicrobials_indicated': 'indication',
        'spectrum_too_broad':       'spectrum',
        'spectrum_too_narrow':      'spectrum',
        'microbiology_mismatch':    'microbiology',
        'incorrect_route':          'route',
        'incorrect_dosing':         'dosing',
        'duration_excessive':       'duration',
        'allergy_mismatch':         'allergies',
    }
    def __init__(self, guideline_text):
        self.raw = guideline_text
        self.sections = self._parse(guideline_text)
    def _parse(self, text):
        kw_map = {
            'indication':   ['indication','antimicrobial','antibiotic','treatment'],
            'spectrum':     ['spectrum','broad','narrow','empiric','coverage'],
            'microbiology': ['culture','organism','sensitivity','microbiology','pathogen'],
            'route':        ['route','intravenous','oral','iv to po','switch'],
            'dosing':       ['dose','dosing','renal','weight','creatinine','eGFR'],
            'duration':     ['duration','days','length','course'],
            'allergies':    ['allergy','allergies','penicillin','hypersensitivity'],
        }
        lines = text.split('\n'); sections = {'full': text[:1200]}
        for sec, kws in kw_map.items():
            rel = []
            for i,line in enumerate(lines):
                if any(k.lower() in line.lower() for k in kws):
                    rel.extend(lines[max(0,i-2):min(len(lines),i+12)])
                    if len(rel) > 80: break
            sections[sec] = '\n'.join(rel[:80]) if rel else sections['full'][:400]
        return sections
    def for_marker(self, marker, max_chars=400):
        k = self.MARKER_SECTION.get(marker,'full')
        return self.sections.get(k, self.sections['full'])[:max_chars]

class EpisodicMemory:
    def __init__(self): self.by_marker_outcome = defaultdict(list)
    def add(self, case):
        for m,v in case.ground_truth.items():
            self.by_marker_outcome[f"{m}_{v}"].append(case)
    def recall(self, marker, outcome, n=2):
        return self.by_marker_outcome.get(f"{marker}_{outcome}",[])[-n:]

class AbstractMemory:
    def __init__(self): self.principles = {}; self._counter = 0
    def add_principle(self, marker, description, supporting_cases):
        pid = f"PRIN_{self._counter:04d}"; self._counter += 1
        self.principles[pid] = AbstractPrinciple(
            principle_id=pid, marker=marker, description=description,
            confidence=0.6, supporting_cases=supporting_cases,
            created_at=datetime.now().isoformat(),
            last_updated=datetime.now().isoformat())
        return pid
    def update_confidence(self, pid, correct):
        if pid in self.principles:
            p = self.principles[pid]
            p.confidence = min(1.0,p.confidence+0.05) if correct else max(0.1,p.confidence-0.1)
            p.last_updated = datetime.now().isoformat()
    def get_for_marker(self, marker, min_confidence=0.5):
        return [p for p in self.principles.values()
                if p.marker==marker and p.confidence>=min_confidence]

class ErrorMemory:
    def __init__(self): self.errors=[]; self.by_marker=defaultdict(list)
    def add(self, case_id, marker, predicted, actual, reflexion):
        e={'case_id':case_id,'marker':marker,'predicted':predicted,
           'actual':actual,'reflexion':reflexion,'timestamp':datetime.now().isoformat()}
        self.errors.append(e); self.by_marker[marker].append(e)
    def get_recent(self, marker, n=2): return self.by_marker[marker][-n:]

class PolicyMemory:
    def __init__(self): self.rules=[]; self._counter=0
    def add(self, condition, action, priority=5):
        self.rules.append(PolicyRule(
            rule_id=f"POL_{self._counter:04d}", condition=condition,
            action=action, priority=priority, success_rate=0.8, usage_count=0))
        self._counter+=1
    def get_for_marker(self, marker):
        return sorted(self.rules,key=lambda r:r.priority,reverse=True)[:2]

class ObservationalMemory:
    def __init__(self): self.counts=defaultdict(lambda:{'YES':0,'NO':0})
    def record(self, marker, prediction, ground_truth):
        self.counts[marker][ground_truth]+=1
    def yes_rate(self, marker):
        c=self.counts[marker]; t=c['YES']+c['NO']
        return c['YES']/t if t>0 else 0.5
    def get_stats(self): return {m:self.yes_rate(m) for m in self.counts}

class ExperienceMemory:
    def __init__(self): self.rules=[]; self.embeddings=None; self.by_marker=defaultdict(list)
    def add(self, rule):
        idx=len(self.rules); self.rules.append(rule); self.by_marker[rule.marker].append(idx)
        if rule.embedding is not None:
            emb=rule.embedding.reshape(1,-1)
            self.embeddings=emb if self.embeddings is None else np.vstack([self.embeddings,emb])
    def retrieve(self, fe, marker, top_k=2, min_confidence=0.30):
        idxs=self.by_marker.get(marker,[])
        if not idxs or self.embeddings is None: return []
        q=fe/(np.linalg.norm(fe)+1e-9); scores=[]
        for idx in idxs:
            r=self.rules[idx]
            if r.confidence<min_confidence: continue
            emb=self.embeddings[idx]/(np.linalg.norm(self.embeddings[idx])+1e-9)
            scores.append((float(q@emb)*(1.0+0.1*min(r.times_prevented_error,3)), idx))
        scores.sort(reverse=True)
        return [self.rules[i] for _,i in scores[:top_k]]
    def update_confidence(self, rule_id, prevented_error):
        for r in self.rules:
            if r.rule_id==rule_id:
                if prevented_error: r.times_prevented_error+=1; r.confidence=min(1.0,r.confidence+0.06)
                else: r.times_contradicted+=1; r.confidence=max(0.1,r.confidence-0.08)
                break
    def prune(self, min_confidence=0.25):
        keep=[i for i,r in enumerate(self.rules) if r.confidence>=min_confidence]
        removed=len(self.rules)-len(keep)
        if removed==0: return 0
        self.rules=[self.rules[i] for i in keep]
        self.embeddings=self.embeddings[keep] if self.embeddings is not None and keep else None
        self.by_marker=defaultdict(list)
        for i,r in enumerate(self.rules): self.by_marker[r.marker].append(i)
        return removed

class CaseSummaryMemory:
    def __init__(self): self.summaries=[]; self.embeddings=None; self.by_marker=defaultdict(list)
    def add(self, summary):
        idx=len(self.summaries); self.summaries.append(summary)
        self.by_marker[summary.marker].append(idx)
        if summary.embedding is not None:
            emb=summary.embedding.reshape(1,-1)
            self.embeddings=emb if self.embeddings is None else np.vstack([self.embeddings,emb])
    def retrieve(self, fe, marker, top_k=2, min_confidence=3.5):
        idxs=self.by_marker.get(marker,[])
        if not idxs or self.embeddings is None: return []
        q=fe/(np.linalg.norm(fe)+1e-9); scores=[]
        for idx in idxs:
            s=self.summaries[idx]
            if s.confidence<min_confidence: continue
            emb=self.embeddings[idx]/(np.linalg.norm(self.embeddings[idx])+1e-9)
            scores.append((float(q@emb)*(1.0+0.05*min(s.times_confirmed,4)), idx))
        scores.sort(reverse=True)
        return [self.summaries[i] for _,i in scores[:top_k]]
    def update_confidence(self, summary_id, confirmed):
        for s in self.summaries:
            if s.summary_id==summary_id:
                if confirmed: s.times_confirmed+=1; s.confidence=min(5.0,s.confidence+0.05)
                else: s.confidence=max(1.0,s.confidence-0.05)
                break
    def prune(self, min_confidence=2.5):
        keep=[i for i,s in enumerate(self.summaries) if s.confidence>=min_confidence]
        removed=len(self.summaries)-len(keep)
        if removed==0: return 0
        self.summaries=[self.summaries[i] for i in keep]
        self.embeddings=self.embeddings[keep] if self.embeddings is not None and keep else None
        self.by_marker=defaultdict(list)
        for i,s in enumerate(self.summaries): self.by_marker[s.marker].append(i)
        return removed


# =============================================================================
# SECTION 4: CHECKPOINT MANAGER (unchanged from V4)
# =============================================================================

class CheckpointManager:
    def __init__(self, checkpoint_dir):
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)

    def save(self, agent, cases_processed, batch_id, training_metrics, metadata=None):
        path = self.checkpoint_dir / f"checkpoint_batch_{batch_id:04d}.pkl"
        state = {
            'cases_processed': cases_processed, 'batch_id': batch_id,
            'training_metrics': training_metrics,
            'timestamp': datetime.now().isoformat(), 'metadata': metadata or {},
            'memories': {
                'short_term': agent.short_term, 'long_term': agent.long_term,
                'episodic': agent.episodic, 'abstract': agent.abstract,
                'error_memory': agent.error_memory, 'policy': agent.policy,
                'observational': agent.observational,
                'experience': agent.experience,
                'case_summaries': agent.case_summaries,
            },
            'counters': {'exp_counter': agent._exp_counter,
                         'sum_counter': agent._sum_counter},
        }
        with open(path,'wb') as f: pickle.dump(state,f)
        latest = self.checkpoint_dir / 'latest_checkpoint.pkl'
        with open(latest,'wb') as f: pickle.dump(state,f)
        print(f"  ✅ Checkpoint saved → {path.name}  (cases: {cases_processed})")
        return path

    def save_checkpoint(self, *a, **kw): return self.save(*a, **kw)

    def load_checkpoint(self, checkpoint_path=None):
        p = Path(checkpoint_path) if checkpoint_path else self.checkpoint_dir/'latest_checkpoint.pkl'
        if not p.exists(): print(f"  ❌ No checkpoint at {p}"); return None
        with open(p,'rb') as f: state=pickle.load(f)
        print(f"  ✅ Loaded — cases:{state['cases_processed']} batch:{state['batch_id']} {state['timestamp'][:19]}")
        return state


# =============================================================================
# SECTION 5: CLINICAL CONSTANTS (markers, defs, bias, policies)
# =============================================================================

MARKERS = [
    'antimicrobials_indicated','spectrum_too_broad','spectrum_too_narrow',
    'microbiology_mismatch','incorrect_route','incorrect_dosing',
    'duration_excessive','allergy_mismatch',
]

MARKER_DEFS = {
    'antimicrobials_indicated': 'Are antimicrobials clinically indicated for this patient?',
    'spectrum_too_broad':       'Is the antibiotic spectrum unnecessarily broad for the likely/confirmed pathogen and CAP severity?',
    'spectrum_too_narrow':      'Is the spectrum inadequate — does it miss a likely or confirmed pathogen for this CAP severity?',
    'microbiology_mismatch':    'Do prescribed antimicrobials mismatch confirmed culture/sensitivity results?',
    'incorrect_route':          'Is the administration route (IV vs PO) inappropriate given clinical status and switch criteria?',
    'incorrect_dosing':         'Is the dose incorrect for this patient given weight, renal function, and guideline recommendations?',
    'duration_excessive':       'Is the effective duration of therapy excessive given CAP severity and clinical status?',
    'allergy_mismatch':         'Do prescribed antimicrobials conflict with documented patient allergies including reaction type?',
}

BIAS_CORRECTIONS = {
    'antimicrobials_indicated': """\
DEFAULT: Antimicrobials ARE indicated unless EXPLICIT evidence of:
  - No radiographic evidence of pneumonia | Non-infective diagnosis confirmed | Treatment completed
If in doubt → YES.""",
    'duration_excessive': """\
Use EFFECTIVE DOT (cumulative days across all antibiotics including IV-to-PO switches).
Thresholds: Mild CAP >5 days | Moderate/Severe CAP >7 days | Legionella >7 days.
Only YES if patient is clinically well AND effective DOT exceeds threshold.""",
    'incorrect_route': """\
IV correct for: severe CAP (CORB≥2), haemodynamically unstable, unable to tolerate oral.
PO correct for: mild CAP or patient meets ALL 7 IV-to-oral switch criteria.
Flag IV as incorrect if all 7 switch criteria are met but patient remains on IV.""",
    'spectrum_too_broad': """\
Check guideline indicator lists by CAP severity. Standard empiric therapy is usually appropriate.
Mild CAP: ceftriaxone, pip-tazo, carbapenems are too broad.
Moderate CAP: carbapenems, vancomycin without MRSA evidence are too broad.
Severe CAP: azithromycin is NOT too broad; ceftriaxone + azithromycin is first line.""",
    'spectrum_too_narrow': """\
Mild CAP: aminoglycosides alone, cefazolin, flucloxacillin, daptomycin are too narrow.
Moderate CAP: same as mild.
Severe CAP: benzylpenicillin alone, doxycycline alone, clarithromycin alone are too narrow.""",
    'allergy_mismatch': """\
Check allergy AND reaction type. Mild penicillin allergy (rash) → avoid amoxicillin, benzylpen, pip-tazo.
Severe penicillin allergy → also avoid ceftriaxone, cefuroxime, carbapenems, aztreonam.
No documented allergy → allergy_mismatch is NO.""",
    'incorrect_dosing': """\
Check: (1) Is drug renally cleared? (2) What is patient's CrCl/renal category?
(3) Compare prescribed dose against guideline table recommendation.
CrCl ≥90 = Normal | 60-89 = Mild | 30-59 = Moderate | <30 = Severe.""",
    'microbiology_mismatch': """\
Only YES if culture result is AVAILABLE AND current antibiotic does NOT cover the organism.
No cultures / pending / negative → microbiology_mismatch is NO.""",
}

DEFAULT_POLICIES = [
    ("CAP diagnosis confirmed","antimicrobials_indicated YES unless contraindication",10),
    ("radiographic consolidation","supports antimicrobial indication",9),
    ("severe CAP / CORB≥2","IV route correct; broad empiric coverage appropriate",8),
    ("penicillin allergy documented","avoid beta-lactams; reaction type determines extent",8),
    ("renal impairment CrCl<90","check dose adjustment against Tables 1-8",7),
    ("atypical pathogen suspected","ensure macrolide or doxycycline included",7),
    ("all 7 IV switch criteria met","IV route is now incorrect — switch to oral",7),
    ("therapy day ≤3","duration_excessive almost always NO at this stage",6),
    ("no culture data available","microbiology_mismatch is almost always NO",6),
    ("patient improving / afebrile","consider IV→PO switch if oral tolerated",6),
]


# =============================================================================
# SECTION 6: LAYER 1 — PYTHON PRE-PROCESSING FUNCTIONS
# Pure Python — no LLM calls. Deterministic clinical calculations.
# =============================================================================

def _cockcroft_gault(age: int, weight_kg: float,
                     creatinine_umol_L: float, sex: str) -> float:
    """CrCl in mL/min using Cockcroft-Gault (guideline standard for dosing)."""
    if age <= 0 or weight_kg <= 0 or creatinine_umol_L <= 0:
        return -1.0
    sex_factor = 1.0 if str(sex).upper().startswith('M') else 0.85
    return ((140 - age) * weight_kg * sex_factor) / (0.815 * creatinine_umol_L)

def _ckd_epi_2021(age: int, creatinine_umol_L: float, sex: str) -> float:
    """eGFR in mL/min/1.73m² using CKD-EPI 2021 (no race variable)."""
    if age <= 0 or creatinine_umol_L <= 0:
        return -1.0
    scr_mg_dl = creatinine_umol_L / 88.42
    female = str(sex).upper().startswith('F')
    kappa  = 0.7 if female else 0.9
    alpha  = -0.241 if female else -0.302
    scr_kappa = scr_mg_dl / kappa
    term1 = min(scr_kappa, 1.0) ** alpha
    term2 = max(scr_kappa, 1.0) ** (-1.200)
    sex_mult = 1.012 if female else 1.0
    return 142 * term1 * term2 * (0.9938 ** age) * sex_mult

def _classify_renal_function(crcl: float, dialysis: str = '') -> str:
    """Maps CrCl to guideline renal category (KEY 3)."""
    d = (dialysis or '').lower()
    if 'ihd' in d or 'haemodialysis' in d or 'hemodialysis' in d:
        return 'IHD'
    if 'crrt' in d or 'cvvh' in d:
        return 'CRRT'
    if 'peritoneal' in d or ' pd' in d or d == 'pd':
        return 'PD'
    if crcl < 0:  return 'Unknown'
    if crcl >= 90: return 'Normal'
    if crcl >= 60: return 'Mild'
    if crcl >= 30: return 'Moderate'
    return 'Severe'

def _normalise_drug(drug_raw: str) -> str:
    """Normalise brand/abbreviation to canonical name matching DOSING_TABLE keys."""
    d = drug_raw.lower().strip()
    # Direct alias lookup
    for alias, canonical in DRUG_ALIASES.items():
        if alias in d:
            return canonical
    # Route inference for common drugs
    route_hint = 'iv' if ' iv' in d or '(iv)' in d else 'po'
    for stem, base in [
        ('amoxicillin-clavulanate','amoxicillin-clavulanate (iv)'),
        ('amoxicillin','amoxicillin (po)' if route_hint=='po' else 'amoxicillin (iv)'),
        ('ceftriaxone','ceftriaxone'),
        ('cefuroxime','cefuroxime (po)'),
        ('cefazolin','cefazolin'),
        ('meropenem','meropenem'),
        ('piperacillin','piperacillin-tazobactam'),
        ('benzylpenicillin','benzylpenicillin'),
        ('flucloxacillin','flucloxacillin (iv)' if route_hint=='iv' else 'flucloxacillin (po)'),
        ('vancomycin','vancomycin (iv)'),
        ('azithromycin','azithromycin (iv)' if route_hint=='iv' else 'azithromycin (po)'),
        ('clarithromycin','clarithromycin (iv)' if route_hint=='iv' else 'clarithromycin (po)'),
        ('doxycycline','doxycycline (po)'),
        ('moxifloxacin','moxifloxacin (iv)' if route_hint=='iv' else 'moxifloxacin (po)'),
        ('levofloxacin','levofloxacin (iv)' if route_hint=='iv' else 'levofloxacin (po)'),
        ('ciprofloxacin','ciprofloxacin (iv)' if route_hint=='iv' else 'ciprofloxacin (po)'),
        ('oseltamivir','oseltamivir (po)'),
        ('metronidazole','metronidazole (po/iv)'),
        ('clindamycin','clindamycin (po/iv)'),
        ('linezolid','linezolid (po/iv)'),
    ]:
        if stem in d:
            return base
    return d  # return as-is if no match

def _calculate_effective_dot(antibiotic_history: List[Dict]) -> int:
    """
    Calculate effective days of therapy (cumulative, not current day number).
    Each dict: {drug, start_day (int), end_day (int or None for 'current')}
    Returns total unique therapy days covered.
    """
    if not antibiotic_history:
        return 0
    all_days: set = set()
    for abx in antibiotic_history:
        try:
            start = int(abx.get('start_day', 1))
            end   = int(abx.get('end_day', abx.get('current_day', start)))
            all_days.update(range(start, end + 1))
        except (TypeError, ValueError):
            pass
    return len(all_days)

def _parse_antibiotic_history(case_text: str) -> List[Dict]:
    """
    Attempt to parse antibiotic history from case text using regex patterns.
    Returns list of {drug, start_day, end_day} dicts.
    Falls back to empty list if parsing fails (LLM extraction handles it).
    """
    history = []
    # Pattern: "drug dose route for X days" or "drug days 1-4"
    patterns = [
        r'(\w[\w\s\-]+?)\s+(?:\d+\s*(?:mg|g|mcg)[\w\s/]*?)\s+(?:iv|po|oral|intravenous)\s+(?:for\s+)?(\d+)\s+days?',
        r'(\w[\w\s\-]+?)\s+days?\s+(\d+)[–\-](\d+)',
        r'(\w[\w\s\-]+?)\s+(?:iv|po)\s+days?\s+(\d+)[–\-](\d+)',
    ]
    for pat in patterns:
        for m in re.finditer(pat, case_text, re.IGNORECASE):
            try:
                drug = m.group(1).strip()
                if len(m.groups()) >= 3 and m.group(3):
                    history.append({'drug': drug,
                                    'start_day': int(m.group(2)),
                                    'end_day':   int(m.group(3))})
                else:
                    history.append({'drug': drug,
                                    'start_day': 1,
                                    'end_day': int(m.group(2))})
            except (IndexError, ValueError):
                continue
    return history

def _score_corb(confusion: bool, sats_lt90: bool,
                rr_gte30: bool, sbp_lt90: bool) -> int:
    return sum([bool(confusion), bool(sats_lt90),
                bool(rr_gte30), bool(sbp_lt90)])

def _score_smart_cop(sbp_lt90: bool, multilobar: bool, albumin_lt35: bool,
                     rr_high: bool, rr_very_high: bool,
                     tachycardia_gt125: bool, confusion: bool,
                     o2_low: bool, o2_very_low: bool, ph_lt735: bool) -> int:
    s = 0
    if sbp_lt90:         s += 2
    if multilobar:       s += 1
    if albumin_lt35:     s += 1
    if rr_very_high:     s += 2
    elif rr_high:        s += 1
    if tachycardia_gt125: s += 1
    if confusion:        s += 1
    if o2_very_low:      s += 2
    elif o2_low:         s += 1
    if ph_lt735:         s += 2
    return s

def _evaluate_iv_oral_switch(vitals: Dict) -> Dict:
    """
    Evaluate all 7 IV-to-oral switch criteria (guideline P159-P166).
    Returns {criterion: bool, all_met: bool, how_many: int, not_met: list}
    """
    c = {
        'clinical_stability_gt24h': bool(vitals.get('clinical_stability_hours', 0) >= 24),
        'afebrile_gt24h':           bool(vitals.get('afebrile_hours', 0) >= 24),
        'heart_rate_lt100':         bool(vitals.get('heart_rate', 999) < 100),
        'resp_rate_lt24':           bool(vitals.get('resp_rate', 999) < 24),
        'sbp_gt90':                 bool(vitals.get('systolic_bp', 0) > 90),
        'sats_stable_improving':    bool(vitals.get('sats_improving', False)),
        'tolerating_oral':          bool(vitals.get('tolerating_oral', False)),
    }
    criteria_list = list(c.values())
    c['all_met']  = all(criteria_list)
    c['how_many'] = sum(criteria_list)
    c['not_met']  = [k for k,v in c.items()
                     if not v and k not in ('all_met','how_many','not_met')]
    return c

def _extract_vitals_from_text(text: str) -> Dict:
    """
    Heuristic extraction of vital signs from raw case text for Layer 1.
    Uses word-boundary anchors and physiological range guards to avoid
    matching numbers from BMI, weight, drug doses, or day numbers.
    Returns dict of vital values for _evaluate_iv_oral_switch().
    """
    t = text.lower()
    vitals: Dict[str, Any] = {}

    # Heart rate — word boundary before keyword, plausible range 30-250 bpm
    for pat in [r'\bheart rate[^\d]*(\d{2,3})', r'\bpulse[^\d]*(\d{2,3})',
                r'\bhr[:\s]+(\d{2,3})', r'\bp[:\s]+(\d{2,3})\s*bpm']:
        hr_m = re.search(pat, t)
        if hr_m:
            val = int(hr_m.group(1))
            if 30 <= val <= 250:
                vitals['heart_rate'] = val
                break

    # Respiratory rate — word boundary, plausible range 8-60 breaths/min
    for pat in [r'\brespiratory rate[^\d]*(\d{1,2})', r'\bresp rate[^\d]*(\d{1,2})',
                r'\brr[:\s]+(\d{1,2})\b', r'\br[:\s]+(\d{1,2})\s*breaths']:
        rr_m = re.search(pat, t)
        if rr_m:
            val = int(rr_m.group(1))
            if 8 <= val <= 60:
                vitals['resp_rate'] = val
                break

    # Systolic BP — require slash for BP format (e.g. 118/76), range 60-250 mmHg
    for pat in [r'\bbp[:\s]*(\d{2,3})[/]', r'\bblood pressure[^\d]*(\d{2,3})[/]',
                r'\bsbp[:\s]*(\d{2,3})']:
        sbp_m = re.search(pat, t)
        if sbp_m:
            val = int(sbp_m.group(1))
            if 60 <= val <= 250:
                vitals['systolic_bp'] = val
                break

    # Afebrile
    if any(s in t for s in ['afebrile','apyrexial','no fever','temp 36','temp 37',
                              'temperature 36','temperature 37']):
        vitals['afebrile_hours'] = 25  # assume >24h if documented
    elif any(s in t for s in ['febrile','fever','temp 38','temp 39','temp 40']):
        vitals['afebrile_hours'] = 0

    # Tolerating oral
    if any(s in t for s in ['tolerating oral','tolerating po','eating and drinking',
                              'able to swallow','tolerating fluids']):
        vitals['tolerating_oral'] = True
    elif any(s in t for s in ['vomiting','not tolerating','unable to swallow','ngt','naso']):
        vitals['tolerating_oral'] = False

    # Sats improving
    if any(s in t for s in ['improving','wean','less oxygen','room air','ra sats']):
        vitals['sats_improving'] = True

    # Clinical stability proxy
    if any(s in t for s in ['clinically stable','clinically improving','improving clinically']):
        vitals['clinical_stability_hours'] = 25

    return vitals

def _lookup_dose(drug_canonical: str, renal_category: str,
                 route: str = '') -> Optional[str]:
    """Look up recommended dose from DOSING_TABLE."""
    key = (drug_canonical.lower(), renal_category)
    return DOSING_TABLE.get(key)

def _drug_too_broad(drug_raw: str, severity: str) -> bool:
    """Check if drug is in the guideline's too-broad list for this CAP severity."""
    d = drug_raw.lower()
    sev = severity.lower() if severity else 'moderate'
    if sev not in TOO_BROAD: sev = 'moderate'
    return any(broad in d for broad in TOO_BROAD.get(sev, set()))

def _drug_too_narrow(drug_raw: str, severity: str) -> bool:
    """Check if drug is in the guideline's too-narrow list for this CAP severity."""
    d = drug_raw.lower()
    sev = severity.lower() if severity else 'moderate'
    if sev not in TOO_NARROW: sev = 'moderate'
    return any(narrow in d for narrow in TOO_NARROW.get(sev, set()))


# =============================================================================
# SECTION 7: MAIN AGENT — V4b
# =============================================================================

class CAPAgentV4b:
    """
    CAP AMS Agent V4b — Three-layer guideline-aware extraction + all V4 memory systems.
    """

    def __init__(self, guideline_text: str,
                 model_name: str = "Qwen/Qwen2.5-3B-Instruct"):

        print("=" * 70)
        print("CAP Agentic AI — V4b")
        print("  Three-Layer Extraction | All V4 Memory Systems")
        print("=" * 70)

        # V3/V4 memory systems
        self.short_term    = ShortTermMemory(capacity=5)
        self.long_term     = LongTermMemory()
        self.semantic      = SemanticMemory(guideline_text)
        self.episodic      = EpisodicMemory()
        self.abstract      = AbstractMemory()
        self.error_memory  = ErrorMemory()
        self.policy        = PolicyMemory()
        self.observational = ObservationalMemory()
        # V4 memory systems
        self.experience      = ExperienceMemory()
        self.case_summaries  = CaseSummaryMemory()
        self._exp_counter    = 0
        self._sum_counter    = 0
        self.debug_mode = False   # Set True via agent.debug_mode=True for per-case debug

        self.training_metrics: List[Dict] = []
        self.markers = MARKERS

        for cond, act, pri in DEFAULT_POLICIES:
            self.policy.add(cond, act, pri)

        print("  Loading sentence embedder...")
        self.embedder = get_embedder()
        print("  ✅ Embedder ready")

        print(f"  Loading LLM: {model_name} ...")
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
            trust_remote_code=True,
        )
        if not torch.cuda.is_available():
            self.model = self.model.to(self.device)
        self.model.eval()

        if torch.cuda.is_available():
            used  = torch.cuda.memory_allocated()/1e9
            total = torch.cuda.get_device_properties(0).total_memory/1e9
            print(f"  VRAM: {used:.1f} / {total:.1f} GB")
        print("  ✅ V4b agent ready\n")

    # ─────────────────────────────────────────────────────────────────────
    # LLM GENERATION (unchanged from V4, single cache clear per case)
    # ─────────────────────────────────────────────────────────────────────

    def embed(self, text: str) -> np.ndarray:
        return self.embedder.encode(text, convert_to_numpy=True, show_progress_bar=False)

    def _generate(self, prompt: str, max_new_tokens: int = 200,
                  system_msg: str = None, token_cap: int = 900) -> str:
        if system_msg is None:
            system_msg = "You are an expert antimicrobial stewardship pharmacist. Be precise and concise."
        messages = [{"role":"system","content":system_msg},
                    {"role":"user","content":prompt}]
        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
        ids = self.tokenizer.encode(text)
        if len(ids) > token_cap:
            ids  = ids[:token_cap]
            text = self.tokenizer.decode(ids, skip_special_tokens=False)
        inputs = self.tokenizer([text], return_tensors="pt").to(self.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=False, temperature=None, top_p=None)
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        if "assistant" in response.lower():
            response = response.split("assistant")[-1].strip()
        del inputs, outputs
        return response

    # ─────────────────────────────────────────────────────────────────────
    # LAYER 1 — PYTHON PRE-PROCESSING
    # Extracts numeric/boolean values from raw case text for Layer 3 annotations.
    # ─────────────────────────────────────────────────────────────────────

    def _layer1_preprocess(self, case_text: str) -> Dict:
        """
        Extract computable clinical values from raw case text.
        Returns a dict of pre-computed values for use in Layer 3 annotation.
        All computation is Python — no LLM call.
        """
        t = case_text.lower()
        results: Dict[str, Any] = {}

        # ── Age ────────────────────────────────────────────────────────────
        age_m = re.search(r'(\d{2,3})[- ]?(?:year|yr)[s -]?(?:old|of age)?', t)
        if not age_m:
            age_m = re.search(r'age[d]?[:\s]+(\d{2,3})', t)
        results['age'] = int(age_m.group(1)) if age_m else None

        # ── Sex ────────────────────────────────────────────────────────────
        if any(s in t for s in ['female','woman','mrs','she/her','she ']):
            results['sex'] = 'F'
        elif any(s in t for s in ['male','man','mr ','he/him','he ']):
            results['sex'] = 'M'
        else:
            results['sex'] = None

        # ── Weight ─────────────────────────────────────────────────────────
        wt_m = re.search(r'(?:weight|wt)[:\s]*(\d{2,3}(?:\.\d)?)\s*kg', t)
        if not wt_m:
            wt_m = re.search(r'(\d{2,3}(?:\.\d)?)\s*kg\b', t)
        results['weight_kg'] = float(wt_m.group(1)) if wt_m else None

        # ── Creatinine / eGFR / CrCl ───────────────────────────────────────
        creat_m = re.search(r'creatinine[:\s]*(\d{2,4}(?:\.\d+)?)\s*(?:umol|µmol|micromol)', t)
        results['creatinine_umol'] = float(creat_m.group(1)) if creat_m else None

        egfr_m = re.search(r'(?:egfr|gfr)[:\s]*(\d{1,3}(?:\.\d+)?)', t)
        results['egfr'] = float(egfr_m.group(1)) if egfr_m else None

        crcl_m = re.search(r'(?:crcl|creatinine clearance|cockcroft)[:\s]*(\d{1,3}(?:\.\d+)?)', t)
        results['crcl_documented'] = float(crcl_m.group(1)) if crcl_m else None

        # ── Calculate CrCl if not directly documented ──────────────────────
        if results.get('crcl_documented'):
            results['crcl'] = results['crcl_documented']
            results['crcl_source'] = 'documented'
        elif all(results.get(k) for k in ['age','weight_kg','creatinine_umol','sex']):
            results['crcl'] = _cockcroft_gault(
                results['age'], results['weight_kg'],
                results['creatinine_umol'], results['sex'])
            results['crcl_source'] = 'calculated_CG'
            # Also calculate eGFR if not documented
            if not results.get('egfr'):
                results['egfr'] = _ckd_epi_2021(
                    results['age'], results['creatinine_umol'], results['sex'])
        else:
            egfr_val = results.get('egfr')
            results['crcl'] = float(egfr_val) if egfr_val is not None else -1.0
            results['crcl_source'] = 'egfr_proxy' if egfr_val is not None else 'unknown'

        # ── Renal category ─────────────────────────────────────────────────
        dialysis_m = re.search(r'(ihd|haemodialysis|hemodialysis|crrt|cvvh|peritoneal dialysis)', t)
        results['dialysis'] = dialysis_m.group(1) if dialysis_m else ''
        results['renal_category'] = _classify_renal_function(
            results.get('crcl', -1.0), results.get('dialysis',''))

        # ── Current antibiotic ─────────────────────────────────────────────
        results['vitals'] = _extract_vitals_from_text(case_text)

        # ── Antibiotic history for DOT calculation ─────────────────────────
        results['antibiotic_history'] = _parse_antibiotic_history(case_text)

        # ── IV-to-oral switch criteria ─────────────────────────────────────
        results['iv_oral_switch'] = _evaluate_iv_oral_switch(results['vitals'])

        # ── CORB component extraction ──────────────────────────────────────
        confusion = any(s in t for s in ['confused','confusion','delirium','altered',
                                          'gcs <','disoriented'])
        sats_lt90 = bool(re.search(r'sat[s]?\s*(?:of\s*)?\s*(?:<|less than)\s*90', t) or
                         re.search(r'8[0-9]%\s*(?:on room air|spo2)', t))
        rr_gte30  = bool(re.search(r'(?:rr|resp rate)\s*(?:of\s*)?\s*(?:≥|>=|>)\s*2[89]|3\d', t))
        sbp_lt90  = bool(re.search(r'(?:bp|sbp)\s*\d*[/\\]\d*|sbp\s*(?:<|less than)\s*90', t) or
                         'hypotensive' in t or 'systolic.*8[0-9]' in t)
        results['corb_components'] = {
            'confusion': confusion, 'sats_lt90': sats_lt90,
            'rr_gte30': rr_gte30, 'sbp_lt90': sbp_lt90,
        }
        results['corb_calculated'] = _score_corb(confusion, sats_lt90, rr_gte30, sbp_lt90)


        # ── Debug output ──────────────────────────────────────────────────
        if getattr(self, 'debug_mode', False):
            print(f"  [L1] Age={results.get('age')} Sex={results.get('sex')} "
                  f"Wt={results.get('weight_kg')}kg")
            print(f"  [L1] Creat={results.get('creatinine_umol')} umol/L | "
                  f"CrCl={results.get('crcl',-1):.1f} mL/min ({results.get('crcl_source','?')})")
            print(f"  [L1] Renal category: {results.get('renal_category','Unknown')}")
            print(f"  [L1] CORB components: {results.get('corb_components',{})} "
                  f"→ score={results.get('corb_calculated','?')}")
            sw = results.get('iv_oral_switch',{})
            print(f"  [L1] IV-oral switch: {sw.get('how_many',0)}/7 criteria met | "
                  f"all_met={sw.get('all_met',False)}")
            history = results.get('antibiotic_history',[])
            print(f"  [L1] Antibiotic history parsed: {len(history)} entries")

        return results

    # ─────────────────────────────────────────────────────────────────────
    # LAYER 2 — GUIDELINE-AWARE LLM EXTRACTION
    # Single LLM call — structured fields guided by guideline requirements.
    # ─────────────────────────────────────────────────────────────────────

    def _layer2_llm_extract(self, case_text: str, precomp: Dict) -> str:
        """
        Guideline-aware structured field extraction.
        Terse prompt — no bracket hints that confuse Qwen into echoing the template.
        Critical fields (drug, route, severity, renal) placed first so they are
        always within the token budget. IV switch criteria terminated explicitly.
        Returns structured field block string.
        """
        # Pre-fill known values from Layer 1 so LLM doesn't need to find them
        age_hint  = str(precomp['age'])      if precomp.get('age')       else 'NOT_STATED'
        sex_hint  = str(precomp['sex'])      if precomp.get('sex')       else 'NOT_STATED'
        wt_hint   = str(precomp['weight_kg'])if precomp.get('weight_kg') else 'NOT_STATED'
        crcl_raw  = precomp.get('crcl_documented') or precomp.get('egfr')
        renal_hint= f"{crcl_raw:.0f} mL/min" if crcl_raw else 'NOT_STATED'
        crat_hint = f"{precomp['creatinine_umol']:.0f} umol/L"                     if precomp.get('creatinine_umol') else 'NOT_STATED'

        prompt = f"""You are extracting structured fields from a clinical case note.
Write the VALUE only after each colon. Write NOT_STATED if the field is absent.
Do NOT repeat field descriptions. Do NOT add extra criteria or numbered lists.

CASE (excerpt):
{case_text[:800]}

Fill each field with the extracted value:
AGE_YEARS: {age_hint}
SEX: {sex_hint}
WEIGHT_KG: {wt_hint}
CREATININE_UMOL: {crat_hint}
RENAL_FUNCTION_DOCUMENTED:
DIALYSIS_TYPE:
CAP_SEVERITY:
CORB_SCORE_AT_ONSET:
SMART_COP_SCORE_AT_ONSET:
IMAGING_IMPRESSION:
NON_INFECTIVE_DIAGNOSIS:
CURRENT_ANTIBIOTIC_NAME:
CURRENT_ANTIBIOTIC_DOSE:
CURRENT_ANTIBIOTIC_FREQUENCY:
CURRENT_ANTIBIOTIC_ROUTE:
PRIOR_ANTIBIOTICS:
CURRENT_THERAPY_DAY:
ALLERGIES_AND_REACTIONS:
MICROBIOLOGY:
SWITCH_CRITERIA_1_STABILITY_GT24H:
SWITCH_CRITERIA_2_AFEBRILE_GT24H:
SWITCH_CRITERIA_3_HR_LT100:
SWITCH_CRITERIA_4_RR_LT24:
SWITCH_CRITERIA_5_SBP_GT90:
SWITCH_CRITERIA_6_SATS_IMPROVING:
SWITCH_CRITERIA_7_TOLERATING_ORAL:
CLINICAL_TRAJECTORY:"""

        response = self._generate(
            prompt, max_new_tokens=320,
            system_msg="Extract clinical fields. Write values only. Stop after CLINICAL_TRAJECTORY.",
            token_cap=650)

        # Parse: keep only lines that look like FIELD: value
        # Exclude lines that are repeating prompt text (contain brackets or are >120 chars)
        lines = []
        for l in response.split('\n'):
            l = l.strip()
            if ':' not in l or len(l) < 4:
                continue
            if '[' in l and ']' in l:          # bracket hints = model echoing template
                continue
            if len(l) > 150:                   # suspiciously long = prose not a field value
                continue
            lines.append(l)
        output = '\n'.join(lines) if lines else response[:500]


        # ── Debug output ──────────────────────────────────────────────────
        if getattr(self, 'debug_mode', False):
            field_count = len([l for l in output.split('\n') if ':' in l])
            not_stated  = output.lower().count('not_stated')
            print(f"  [L2] LLM extraction: {len(output)} chars | "
                  f"{field_count} fields | {not_stated} NOT_STATED")
            # Check critical fields
            for cf in ['CURRENT_ANTIBIOTIC_NAME','CORB_SCORE_AT_ONSET',
                       'CURRENT_THERAPY_DAY','ALLERGIES_AND_REACTIONS',
                       'MICROBIOLOGY','RENAL_FUNCTION_DOCUMENTED']:
                present = cf.lower() in output.lower()
                icon = "✅" if present else "⚠️ MISSING"
                print(f"  [L2]   {icon} {cf}")

        return output

        # ── Debug output ──────────────────────────────────────────────────
        if getattr(self, 'debug_mode', False):
            field_count = len([l for l in output.split('\n') if ':' in l])
            not_stated  = output.lower().count('not_stated')
            print(f"  [L2] LLM extraction: {len(output)} chars | "
                  f"{field_count} fields | {not_stated} NOT_STATED")
            # Check critical fields
            for cf in ['CURRENT_ANTIBIOTIC_NAME','CORB_SCORE_AT_ONSET',
                       'CURRENT_THERAPY_DAY','DOCUMENTED_ALLERGIES',
                       'MICROBIOLOGY_RESULTS','RENAL_FUNCTION_AS_DOCUMENTED']:
                present = cf.lower() in output.lower()
                icon = "✅" if present else "⚠️ MISSING"
                print(f"  [L2]   {icon} {cf}")

        return output

    # ─────────────────────────────────────────────────────────────────────
    # LAYER 3 — PYTHON GUIDELINE FLAG ANNOTATION
    # Combines Layer 1 precomputed values + Layer 2 LLM extraction.
    # Generates GUIDELINE_FLAG annotations using guideline indicators directly.
    # No LLM call.
    # ─────────────────────────────────────────────────────────────────────

    def _layer3_annotate(self, llm_fields: str, precomp: Dict,
                         case_text: str) -> str:
        """
        Append GUIDELINE_FLAG annotations to the LLM-extracted field block.
        All logic derived from guideline inappropriate prescribing indicators.
        """
        fs    = llm_fields.lower()
        ct    = case_text.lower()
        flags = []

        # ── Renal function annotation ──────────────────────────────────────
        renal_cat = precomp.get('renal_category', 'Unknown')
        _crcl_raw = precomp.get('crcl')
        crcl      = float(_crcl_raw) if (_crcl_raw is not None and _crcl_raw > 0) else -1.0
        crcl_src  = precomp.get('crcl_source', 'unknown')
        if renal_cat != 'Unknown':
            crcl_str = f"{crcl:.1f} mL/min" if crcl > 0 else "not calculable"
            flags.append(
                f"RENAL_CATEGORY: {renal_cat} "
                f"(CrCl={crcl_str}, source={crcl_src})")

        # ── CAP severity determination ─────────────────────────────────────
        severity = 'moderate'  # safe default
        _sev_m = re.search(r'cap_severity[:\s]*([^\n\[]{1,30})', fs)
        if _sev_m:
            _sev_val = _sev_m.group(1).lower().strip()
            if 'mild' in _sev_val:
                severity = 'mild'
            elif 'severe' in _sev_val:
                severity = 'severe'
            elif 'moderate' in _sev_val:
                severity = 'moderate'
        # Cross-check with CORB
        corb_calc = precomp.get('corb_calculated', -1)
        corb_m    = re.search(r'corb_score_at_onset[:\s]*([0-4])', fs)
        corb_doc  = int(corb_m.group(1)) if corb_m else None
        corb      = corb_doc if corb_doc is not None else corb_calc
        if corb >= 2:
            severity = 'severe'
            flags.append(f"GUIDELINE_FLAG: CORB={corb} at onset → Severe CAP")
        elif corb == 1:
            if severity == 'mild':
                flags.append(f"GUIDELINE_FLAG: CORB={corb} at onset → Moderate CAP")
            severity = max(severity, 'moderate',
                           key=lambda s: ['mild','moderate','severe'].index(s))
        elif corb == 0:
            flags.append(f"GUIDELINE_FLAG: CORB=0 at onset → Mild CAP")
            if severity != 'mild': severity = 'mild'

        # ── Current antibiotic extraction from LLM fields ──────────────────
        drug_raw  = ''
        route_raw = 'iv'  # safe default
        drug_m = re.search(r'current_antibiotic_name[:\s]*([^\n\[]+)', fs)
        if drug_m:
            drug_raw = drug_m.group(1).strip().rstrip(':').strip()
            # Discard if it looks like a template echo (contains field name words)
            if any(w in drug_raw for w in ['antibiotic','frequency','route','criteria']):
                drug_raw = ''
        route_m = re.search(r'current_antibiotic_route[:\s]*([^\n\[]+)', fs)
        if route_m:
            route_raw = route_m.group(1).strip().lower()
            if '[' in route_raw or 'or po' in route_raw or 'or iv' in route_raw:
                route_raw = 'iv'  # reset to safe default if template echo
        dose_m = re.search(r'current_antibiotic_dose[^\n]*[:\s]*([^\n\[]+)', fs)
        dose_raw = dose_m.group(1).strip() if dose_m else ''
        if dose_raw and '[' in dose_raw: dose_raw = ''  # discard template echo

        drug_canonical = _normalise_drug(drug_raw) if drug_raw else ''
        is_iv = 'iv' in route_raw or 'intravenous' in route_raw

        # ── Effective DOT calculation ──────────────────────────────────────
        # Try to parse prior antibiotics from LLM extraction
        prior_abx_block_m = re.search(
            r'prior_antibiotics_with_days[:\s]*(.*?)(?=\n[A-Z_]+:|\Z)',
            llm_fields, re.DOTALL | re.IGNORECASE)
        effective_dot = 0
        dot_source    = 'unknown'

        if prior_abx_block_m:
            block = prior_abx_block_m.group(1).strip()
            history = []
            for line in block.split('\n'):
                day_range = re.search(r'day\s*(\d+)[–\-](\d+)', line, re.IGNORECASE)
                day_start = re.search(r'day\s*(\d+)', line, re.IGNORECASE)
                cur_day_m = re.search(r'current_therapy_day[:\s]*(\d+)', fs)
                current_day = int(cur_day_m.group(1)) if cur_day_m else None
                if day_range:
                    history.append({'start_day': int(day_range.group(1)),
                                    'end_day':   int(day_range.group(2))})
                elif day_start and current_day:
                    history.append({'start_day': int(day_start.group(1)),
                                    'end_day':   current_day})
            if history:
                effective_dot = _calculate_effective_dot(history)
                dot_source = 'parsed_from_history'

        if effective_dot == 0:
            # Fallback: use current day number as lower bound
            cur_day_m = re.search(r'current_therapy_day[:\s]*(\d+)', fs)
            if cur_day_m:
                effective_dot = int(cur_day_m.group(1))
                dot_source = 'current_day_only'

        if effective_dot > 0:
            threshold = DURATION_THRESHOLD.get(severity, 7)
            flags.append(f"EFFECTIVE_DOT: {effective_dot} days (source: {dot_source})")

            # Check if Legionella
            if any(s in fs or s in ct for s in ['legionella','legionnaire']):
                threshold = 7
                flags.append("GUIDELINE_FLAG: Legionella — duration threshold is 7 days")

            # Clinical trajectory
            traj_m = re.search(r'clinical_trajectory[:\s]*([^\n\[]+)', fs)
            traj = traj_m.group(1).strip().lower() if traj_m else ''
            if effective_dot > threshold and ('improv' in traj or 'well' in traj):
                flags.append(
                    f"GUIDELINE_FLAG: Effective DOT={effective_dot}d > {threshold}d threshold "
                    f"for {severity} CAP and patient is improving → duration_excessive likely YES")
            elif effective_dot > threshold:
                flags.append(
                    f"GUIDELINE_FLAG: Effective DOT={effective_dot}d exceeds {threshold}d "
                    f"threshold for {severity} CAP — assess clinical status for duration_excessive")
            elif effective_dot <= 3:
                flags.append(
                    f"GUIDELINE_FLAG: Effective DOT={effective_dot}d — "
                    f"duration_excessive is NO at this stage")

        # ── Route assessment ───────────────────────────────────────────────
        switch = precomp.get('iv_oral_switch', {})
        if is_iv:
            if severity == 'mild':
                flags.append(
                    "GUIDELINE_FLAG: IV route for mild CAP → incorrect_route is YES "
                    "(guideline P95: IV is incorrect for mild CAP)")
            elif switch.get('all_met'):
                flags.append(
                    "GUIDELINE_FLAG: All 7 IV-to-oral switch criteria met but patient "
                    "remains on IV → incorrect_route is YES (guideline P179)")
            elif switch.get('how_many', 0) >= 5:
                not_met = ', '.join(switch.get('not_met',[])[:3])
                flags.append(
                    f"GUIDELINE_FLAG: {switch['how_many']}/7 switch criteria met. "
                    f"Not yet met: {not_met} — review incorrect_route")
            else:
                flags.append(
                    f"GUIDELINE_FLAG: Patient on IV, {switch.get('how_many',0)}/7 "
                    f"switch criteria met — IV may still be appropriate")
        else:  # PO route
            if severity in ('moderate','severe'):
                traj_m = re.search(r'clinical_trajectory[:\s]*([^\n\[]+)', fs)
                traj = traj_m.group(1).strip().lower() if traj_m else ''
                if 'deterior' in traj or 'unwell' in traj or 'unstable' in traj:
                    flags.append(
                        "GUIDELINE_FLAG: PO route for moderate/severe CAP with clinical "
                        "deterioration → incorrect_route may be YES (guideline P180)")

        # ── Spectrum assessment ────────────────────────────────────────────
        if drug_raw:
            if _drug_too_broad(drug_raw, severity):
                flags.append(
                    f"GUIDELINE_FLAG: {drug_raw} is listed as inappropriately BROAD "
                    f"for {severity} CAP in guideline indicators → "
                    f"spectrum_too_broad likely YES")
            elif _drug_too_narrow(drug_raw, severity):
                flags.append(
                    f"GUIDELINE_FLAG: {drug_raw} is listed as TOO NARROW "
                    f"for {severity} CAP in guideline indicators → "
                    f"spectrum_too_narrow likely YES")
            else:
                flags.append(
                    f"GUIDELINE_FLAG: {drug_raw} spectrum appears appropriate "
                    f"for {severity} CAP based on guideline indicator lists")

        # ── Dosing assessment (Tables 1-8) ─────────────────────────────────
        if drug_canonical and renal_cat != 'Unknown':
            recommended = _lookup_dose(drug_canonical, renal_cat)
            if recommended:
                if drug_canonical in NO_RENAL_ADJUSTMENT:
                    flags.append(
                        f"GUIDELINE_FLAG: {drug_canonical} does NOT require renal "
                        f"adjustment — compare prescribed dose {dose_raw} against "
                        f"standard: {recommended}")
                else:
                    flags.append(
                        f"GUIDELINE_FLAG: For {drug_canonical} with {renal_cat} renal "
                        f"function (CrCl≈{crcl:.0f} mL/min), guideline recommends: "
                        f"{recommended}. Prescribed: {dose_raw} — compare for incorrect_dosing")
            else:
                flags.append(
                    f"GUIDELINE_FLAG: No dosing table entry found for {drug_canonical} — "
                    f"manual review required for incorrect_dosing")

        # ── Allergy assessment ─────────────────────────────────────────────
        allergy_m = re.search(r'(?:allergies_and_reactions|documented_allergies)[^\n]*[:\s]*([^\n\[]+)', fs)
        allergy_raw = allergy_m.group(1).strip() if allergy_m else ''
        if not allergy_raw or any(s in allergy_raw for s in
                                   ['nkda','none','nil','no known','not_stated']):
            flags.append("GUIDELINE_FLAG: No documented allergy → allergy_mismatch is NO")
        else:
            flags.append(f"GUIDELINE_FLAG: Documented allergy — {allergy_raw}")
            # Check penicillin cross-reactivity
            if 'penicillin' in allergy_raw or 'amoxicillin' in allergy_raw:
                reaction = 'rash'
                if any(s in allergy_raw for s in
                       ['anaphylaxis','anaphylactic','severe','urticaria','angioedema']):
                    reaction = 'anaphylaxis'
                elif any(s in allergy_raw for s in ['severe','immediate','type 1']):
                    reaction = 'severe'
                contraindicated = PEN_ALLERGY_CONTRAINDICATED.get(reaction, set())
                if drug_raw and any(c in drug_raw.lower() for c in contraindicated):
                    flags.append(
                        f"GUIDELINE_FLAG: ⚠️ ALLERGY MISMATCH — Penicillin "
                        f"allergy ({reaction}) + {drug_raw} prescribed → "
                        f"allergy_mismatch is YES")

        # ── Microbiology assessment ────────────────────────────────────────
        micro_m = re.search(r'microbiology(?:_results)?[:\s]*([^\n\[]+)', fs)
        micro = micro_m.group(1).strip() if micro_m else ''
        if not micro or any(s in micro for s in ['n/a','negative','no growth',
                                                   'pending','not_stated','no culture']):
            flags.append(
                "GUIDELINE_FLAG: No culture result / negative → "
                "microbiology_mismatch is almost always NO")
        else:
            flags.append(
                f"GUIDELINE_FLAG: Culture available — {micro[:120]} — "
                f"check whether {drug_raw} covers this organism")

        # ── Antimicrobials indicated assessment ────────────────────────────
        imaging_m = re.search(r'(?:imaging_impression|diagnosis_and_imaging)[^\n]*[:\s]*([^\n\[]+)', fs)
        imaging = imaging_m.group(1).strip() if imaging_m else ''
        non_inf_m = re.search(r'non_infective(?:_diagnosis)?[^\n]*[:\s]*([^\n\[]+)', fs)
        non_inf = non_inf_m.group(1).strip() if non_inf_m else ''
        if any(s in imaging.lower() for s in
               ['consolidation','pneumonia','infiltrate','opacification','cap']):
            flags.append(
                "GUIDELINE_FLAG: Imaging confirms pneumonia → "
                "antimicrobials_indicated is YES")
        elif any(s in non_inf.lower() for s in
                 ['heart failure','copd','pulmonary embolism','pe ','viral']):
            flags.append(
                f"GUIDELINE_FLAG: Evidence of non-infective diagnosis ({non_inf[:80]}) — "
                f"review antimicrobials_indicated")
        elif not imaging or 'not_stated' in imaging.lower():
            flags.append(
                "GUIDELINE_FLAG: Imaging impression not clearly documented — "
                "confirm radiographic evidence of pneumonia for antimicrobials_indicated")


        # ── Debug output ──────────────────────────────────────────────────
        if getattr(self, 'debug_mode', False):
            print(f"  [L3] Generated {len(flags)} guideline flags")
            for fg in flags:
                icon = "🚨" if "likely yes" in fg.lower() or "mismatch" in fg.lower() else "ℹ️ "
                print(f"  [L3]   {icon} {fg[:100]}")

        # ── Combine ────────────────────────────────────────────────────────
        flag_block = '\n'.join(flags)
        return llm_fields + '\n\nGUIDELINE_CONTEXT:\n' + flag_block

    # ─────────────────────────────────────────────────────────────────────
    # FULL EXTRACTION PIPELINE (Layers 1+2+3)
    # ─────────────────────────────────────────────────────────────────────

    def _extract_features(self, case_text: str) -> Tuple[str, Dict]:
        """
        Run the three-layer extraction pipeline.
        Returns (feature_summary_string, precomp_dict).
        """
        precomp     = self._layer1_preprocess(case_text)
        llm_fields  = self._layer2_llm_extract(case_text, precomp)
        feature_sum = self._layer3_annotate(llm_fields, precomp, case_text)
        return feature_sum, precomp

    # ─────────────────────────────────────────────────────────────────────
    # DECISION PROMPT (unchanged from V4)
    # ─────────────────────────────────────────────────────────────────────

    def _build_decision_prompt(self, feature_summary: str, marker: str,
                               feature_embedding: np.ndarray,
                               use_memory: bool) -> Tuple[str, List, List]:
        guideline   = self.semantic.for_marker(marker, max_chars=350)
        rules_text  = '\n'.join(
            f"  • If {r.condition} → {r.action}"
            for r in self.policy.get_for_marker(marker))
        yes_rate = self.observational.yes_rate(marker)
        if yes_rate < 0.15:
            obs_note = f"⚠️ Only predicted YES {yes_rate:.0%} — check for under-prediction."
        elif yes_rate > 0.85:
            obs_note = f"⚠️ Predicted YES {yes_rate:.0%} — ensure evidence-based."
        else:
            obs_note = f"Your YES rate: {yes_rate:.0%}"

        bias = BIAS_CORRECTIONS.get(marker,'')
        retrieved_rules, retrieved_summaries = [], []
        exp_block = sum_block = ''

        if use_memory:
            retrieved_rules = self.experience.retrieve(
                feature_embedding, marker, top_k=2, min_confidence=0.30)
            if retrieved_rules:
                lines = []
                for r in retrieved_rules:
                    r.times_retrieved += 1
                    lines.append(f"  ⚠️ LEARNT RULE [{r.confidence:.0%}]: "
                                 f"{r.clinical_rule[:120]}\n"
                                 f"     Watch for: {', '.join(r.trigger_features[:3])}")
                exp_block = '<learnt_rules>\n' + '\n'.join(lines) + '\n</learnt_rules>'

            retrieved_summaries = self.case_summaries.retrieve(
                feature_embedding, marker, top_k=2, min_confidence=3.5)
            if retrieved_summaries:
                lines = []
                for s in retrieved_summaries:
                    s.times_retrieved += 1
                    lines.append(f"  ✓ SIMILAR CASE [{s.prediction}]: "
                                 f"{s.case_pattern[:100]}\n"
                                 f"     Why {s.prediction}: {s.why_prediction[:100]}")
                sum_block = '<similar_correct_cases>\n' + '\n'.join(lines) + '\n</similar_correct_cases>'

        prompt = f"""Assess marker <{marker}> for this clinical case.

<structured_case>
{feature_summary[:800]}
</structured_case>

<guideline>
{guideline}
</guideline>

<clinical_rules>
{rules_text}
</clinical_rules>

<marker_definition>
{marker}: {MARKER_DEFS[marker]}
</marker_definition>

<bias_guidance>
{bias}
{obs_note}
</bias_guidance>

{exp_block}

{sum_block}

Respond EXACTLY in this format (two lines only):
CONFIDENCE: [1=very unsure  2=unsure  3=moderate  4=confident  5=very confident]
PREDICTION: [YES or NO]"""

        return prompt, retrieved_rules, retrieved_summaries

    def _parse_decision(self, response: str) -> Tuple[str, int]:
        prediction, confidence = 'NO', 3
        if 'PREDICTION:' in response:
            pred_line = response.split('PREDICTION:')[-1].strip().split('\n')[0]
            if 'YES' in pred_line.upper():
                prediction = 'YES'
        if 'CONFIDENCE:' in response:
            conf_line = response.split('CONFIDENCE:')[-1].strip().split('\n')[0]
            m = re.search(r'[1-5]', conf_line)
            if m: confidence = int(m.group())
        return prediction, confidence

    # ─────────────────────────────────────────────────────────────────────
    # PYTHON SANITY CHECK (expanded — uses precomp values)
    # ─────────────────────────────────────────────────────────────────────

    def _sanity_check(self, marker: str, prediction: str,
                      feature_summary: str, case_text: str,
                      precomp: Dict = None) -> str:
        fs  = feature_summary.lower()
        ct  = case_text.lower()
        pre = precomp or {}

        if marker == 'antimicrobials_indicated':
            if prediction == 'NO':
                if any(s in fs or s in ct for s in
                       ['consolidation','pneumonia','infiltrate','cap','opacification']):
                    return 'YES'

        elif marker == 'microbiology_mismatch':
            if not any(s in fs for s in
                       ['organism','culture positive','susceptib','sensit',
                        'resistant','no growth']):
                return 'NO'

        elif marker == 'allergy_mismatch':
            if any(s in fs for s in ['nkda','no known drug allergy','no allerg',
                                      'no documented allerg']):
                return 'NO'
            if 'allergy mismatch is no' in fs:
                return 'NO'
            if 'allergy mismatch is yes' in fs or 'allergy mismatch — ' in fs:
                return 'YES'

        elif marker == 'duration_excessive':
            if 'dot=1d' in fs or 'dot=2d' in fs or 'dot=3d' in fs:
                return 'NO'
            if 'effective dot' in fs and 'likely yes' in fs and prediction == 'NO':
                return 'YES'

        elif marker == 'incorrect_route':
            if 'iv route for mild cap → incorrect_route is yes' in fs:
                return 'YES'
            if 'all 7 iv-to-oral switch criteria met but patient remains on iv' in fs:
                return 'YES'
            corb = pre.get('corb_calculated', -1)
            is_iv = any(s in fs for s in ['route: iv','route:iv','intravenous'])
            if corb >= 2 and is_iv:
                return 'NO'

        elif marker == 'spectrum_too_broad':
            if 'spectrum_too_broad likely yes' in fs and prediction == 'NO':
                return 'YES'

        elif marker == 'spectrum_too_narrow':
            if 'spectrum_too_narrow likely yes' in fs and prediction == 'NO':
                return 'YES'

        elif marker == 'incorrect_dosing':
            if 'no dosing table entry' in fs:
                return prediction  # cannot determine — leave to LLM

        return prediction

    # ─────────────────────────────────────────────────────────────────────
    # REFLEXION + EXPERIENCE RULE (unchanged from V4)
    # ─────────────────────────────────────────────────────────────────────

    def _generate_batch_reflexion(self, feature_summary: str,
                                  errors: Dict) -> Dict[str, str]:
        error_lines = [f"  - {m}: predicted={v[0]}, correct={v[1]}"
                       for m, v in errors.items()]
        prompt = f"""You predicted incorrectly for these markers:
{chr(10).join(error_lines)}

Case features:
{feature_summary[:400]}

For EACH marker, complete on ONE line:
MARKER | CLINICAL_RULE: [IF/THEN rule for correct answer] | TRIGGER: [key features] | MISTAKE: [missed_feature|misapplied_guideline|ignored_culture|wrong_severity|dosing_arithmetic]"""

        response = self._generate(
            prompt, max_new_tokens=200,
            system_msg="Generate concise IF/THEN clinical rules. One line per marker.")
        result = {m: f"Error on {m}" for m in errors}
        for line in response.split('\n'):
            if '|' not in line: continue
            for marker in errors:
                if marker in line.lower() or marker.replace('_',' ') in line.lower():
                    result[marker] = line[:200]; break
        return result

    def _parse_experience_rule(self, line: str, marker: str) -> Tuple[str, List[str], str]:
        rule = line[:200]; feats = []; mtype = 'missed_feature'
        if 'CLINICAL_RULE:' in line:
            rule = line.split('CLINICAL_RULE:')[-1].split('|')[0].strip()[:200]
        if 'TRIGGER:' in line:
            trig = line.split('TRIGGER:')[-1].split('|')[0].strip()
            feats = [f.strip() for f in trig.split(',')][:3]
        if 'MISTAKE:' in line:
            mt = line.split('MISTAKE:')[-1].strip().split()[0].lower()
            if mt in {'missed_feature','misapplied_guideline','ignored_culture',
                      'wrong_severity','dosing_arithmetic'}:
                mtype = mt
        if not feats:
            feats = [kw for kw in ['corb','iv','oral','culture','allerg','renal',
                                    'dose','day','severity','spectrum']
                     if kw in rule.lower()][:3] or [marker]
        return rule, feats, mtype

    def _generate_case_summary(self, feature_summary: str,
                               marker: str, prediction: str) -> Dict:
        prompt = f"""You correctly assessed {marker} = {prediction}.
Case: {feature_summary[:350]}

Complete EXACTLY (one line each):
CASE_PATTERN: [one sentence - key clinical pattern]
WHY_{prediction}: [one sentence - specific evidence making {marker}={prediction} correct]
DISTINGUISHING: [2-3 key features separated by semicolons]"""

        response = self._generate(prompt, max_new_tokens=100,
                                  system_msg="Summarise correct clinical reasoning concisely.")
        result = {'case_pattern': f"{marker}={prediction} case",
                  'why_prediction': f"Correct {prediction}",
                  'distinguishing_features': [marker]}
        for line in response.split('\n'):
            line = line.strip()
            if line.startswith('CASE_PATTERN:'):
                result['case_pattern'] = line.split(':',1)[-1].strip()[:180]
            elif line.startswith(f'WHY_{prediction}:'):
                result['why_prediction'] = line.split(':',1)[-1].strip()[:180]
            elif line.startswith('DISTINGUISHING:'):
                feats = line.split(':',1)[-1].strip()
                result['distinguishing_features'] = [f.strip() for f in feats.split(';')][:3]
        return result

    def _extract_principles(self, cases_window: List[CaseMemory]):
        for marker in self.markers:
            errors = [c for c in cases_window
                      if c.predictions.get(marker) != c.ground_truth.get(marker)]
            if len(errors) < 3: continue
            fn = sum(1 for e in errors
                     if e.predictions.get(marker)=='NO' and e.ground_truth.get(marker)=='YES')
            fp = sum(1 for e in errors
                     if e.predictions.get(marker)=='YES' and e.ground_truth.get(marker)=='NO')
            if fn == 0 and fp == 0: continue
            direction = "under-predict YES" if fn >= fp else "over-predict YES"
            prompt = f"""For marker '{marker}', last {len(cases_window)} cases:
False negatives (predicted NO, was YES): {fn}
False positives (predicted YES, was NO): {fp}
Tendency: {direction}
Complete in ONE line:
PRINCIPLE: I tend to {direction} for {marker} because __ and I should instead __
"""
            response = self._generate(prompt, max_new_tokens=80,
                                      system_msg="Identify prediction bias in one sentence.")
            desc = f"[{marker}] Tends to {direction}."
            for line in response.split('\n'):
                if 'PRINCIPLE:' in line:
                    desc = line.split('PRINCIPLE:')[-1].strip()[:200]; break
            self.abstract.add_principle(marker, desc, [c.case_id for c in errors[:5]])

    # ─────────────────────────────────────────────────────────────────────
    # ASSESS ONE CASE — full V4b pipeline
    # ─────────────────────────────────────────────────────────────────────

    def assess_case(self, case_id: str, case_text: str,
                    ground_truth: Dict[str, str],
                    use_memory: bool = True) -> CaseMemory:


        if getattr(self, 'debug_mode', False):
            print(f"\n{'═'*60}")
            print(f"  CASE: {case_id}")
            print(f"{'═'*60}")

        # Three-layer extraction
        feature_summary, precomp = self._extract_features(case_text)
        feature_embedding = self.embed(feature_summary)

        predictions: Dict[str, str] = {}
        confidences: Dict[str, int] = {}
        all_retrieved_rules:    Dict[str, List] = {}
        all_retrieved_summaries: Dict[str, List] = {}

        # Per-marker decision calls
        for marker in self.markers:
            prompt, ret_rules, ret_sums = self._build_decision_prompt(
                feature_summary, marker, feature_embedding, use_memory)
            response   = self._generate(prompt, max_new_tokens=30)
            pred, conf = self._parse_decision(response)
            pred       = self._sanity_check(marker, pred, feature_summary,
                                            case_text, precomp)
            predictions[marker]  = pred
            confidences[marker]  = conf
            all_retrieved_rules[marker]     = ret_rules
            all_retrieved_summaries[marker] = ret_sums

            if getattr(self, 'debug_mode', False):
                gt_val = ground_truth.get(marker, 'NO')
                correct_icon = "✅" if pred == gt_val else "❌"
                sanity_note = f" [sanity→{pred}]" if pred != predictions.get(marker+'_pre_sanity', pred) else ""
                print(f"  [DEC] {marker:<35} "
                      f"pred={pred} conf={conf} gt={gt_val} {correct_icon}{sanity_note}")

            self.observational.record(marker, pred, ground_truth.get(marker,'NO'))

        # Identify errors and correct high-confidence predictions
        errors_this_case: Dict = {}
        correct_hc:       Dict = {}
        for marker in self.markers:
            gt   = ground_truth.get(marker,'NO')
            pred = predictions[marker]
            conf = confidences[marker]
            if pred != gt:
                if conf >= 3:
                    errors_this_case[marker] = (pred, gt, conf, '')
            else:
                if conf >= 4:
                    correct_hc[marker] = conf

        reflexion_notes: Dict[str, str] = {m:'' for m in self.markers}

        # Batched reflexion → ExperienceMemory
        if errors_this_case:
            batch_refs = self._generate_batch_reflexion(feature_summary, errors_this_case)
            for marker, line in batch_refs.items():
                gt = ground_truth.get(marker,'NO')
                reflexion_notes[marker] = line
                self.error_memory.add(case_id, marker, predictions[marker], gt, line)
                rule_text, trigger_feats, mistake_type = \
                    self._parse_experience_rule(line, marker)
                rule_emb = self.embed(rule_text[:100]+' '+' '.join(trigger_feats))
                self.experience.add(ExperienceRule(
                    rule_id=f"EXP_{self._exp_counter:05d}", marker=marker,
                    clinical_rule=rule_text, trigger_features=trigger_feats,
                    mistake_type=mistake_type, source_case_id=case_id,
                    embedding=rule_emb, confidence=0.50,
                    created_at=datetime.now().isoformat()))
                self._exp_counter += 1
                for r in all_retrieved_rules.get(marker,[]):
                    self.experience.update_confidence(r.rule_id, prevented_error=False)

        # CaseSummaryMemory (cap at 3 per case)
        MAX_SUMMARIES = 3
        sum_count = 0
        for marker, conf in sorted(correct_hc.items(), key=lambda x:x[1], reverse=True):
            if sum_count >= MAX_SUMMARIES: break
            pred = predictions[marker]
            sd   = self._generate_case_summary(feature_summary, marker, pred)
            emb  = self.embed(sd['case_pattern']+' '+' '.join(sd['distinguishing_features']))
            self.case_summaries.add(CaseSummaryEntry(
                summary_id=f"SUM_{self._sum_counter:05d}", marker=marker,
                prediction=pred, case_pattern=sd['case_pattern'],
                why_prediction=sd['why_prediction'],
                distinguishing_features=sd['distinguishing_features'],
                source_case_id=case_id, embedding=emb, confidence=float(conf),
                created_at=datetime.now().isoformat()))
            self._sum_counter += 1
            for s in all_retrieved_summaries.get(marker,[]):
                self.case_summaries.update_confidence(s.summary_id, confirmed=True)
            sum_count += 1

        # Metrics
        correct  = sum(1 for m in self.markers
                       if predictions.get(m)==ground_truth.get(m))
        accuracy = correct / len(self.markers)
        marker_metrics = {m: 1.0 if predictions.get(m)==ground_truth.get(m) else 0.0
                          for m in self.markers}

        case_mem = CaseMemory(
            case_id=case_id, case_text=case_text,
            feature_summary=feature_summary,
            predictions=predictions, ground_truth=ground_truth,
            confidences=confidences, reasoning={m:'' for m in self.markers},
            reflexion_notes=reflexion_notes,
            timestamp=datetime.now().isoformat(),
            performance_metrics={'accuracy':accuracy,'correct':correct,
                                 'total':len(self.markers), **marker_metrics},
            embedding=feature_embedding)


        if getattr(self, 'debug_mode', False):
            n_errors   = len(errors_this_case)
            n_correct_hc = len(correct_hc)
            print(f"  [CASE] Accuracy={accuracy:.3f} ({correct}/{len(self.markers)}) | "
                  f"Errors→reflexion={n_errors} | CorrectHC→summary={n_correct_hc}")
            if errors_this_case:
                print(f"  [CASE] Error markers: {list(errors_this_case.keys())}")

        self.short_term.add(case_mem)
        self.long_term.add(case_mem)
        self.episodic.add(case_mem)
        torch.cuda.empty_cache()
        return case_mem

    # ─────────────────────────────────────────────────────────────────────
    # TRAINING LOOP (unchanged from V4)
    # ─────────────────────────────────────────────────────────────────────

    def train(self, train_cases, batch_size=50, checkpoint_mgr=None,
              start_from_batch=0, max_batches=None,
              auto_save_every=1, prune_every=200):

        print(f"\n{'='*70}")
        print(f"V4b TRAINING  |  {len(train_cases)} cases  |  batch_size={batch_size}")
        print(f"{'='*70}\n")
        total_batches = (len(train_cases)+batch_size-1)//batch_size
        window: List[CaseMemory] = []
        cases_since_abstraction = total_cases = 0

        for batch_idx in range(start_from_batch, total_batches):
            if max_batches and batch_idx >= start_from_batch+max_batches:
                print(f"\n⏸  max_batches={max_batches} reached — stopping."); break

            b_start = batch_idx*batch_size
            b_end   = min(b_start+batch_size, len(train_cases))
            batch   = train_cases[b_start:b_end]

            print(f"\n{'─'*60}")
            print(f"Batch {batch_idx+1}/{total_batches}  |  cases {b_start}–{b_end-1}")
            print(f"  Experience rules : {len(self.experience.rules)}")
            print(f"  Case summaries   : {len(self.case_summaries.summaries)}")
            print(f"{'─'*60}")

            batch_cases = []
            for case_data in tqdm(batch, desc=f"Batch {batch_idx+1}"):
                cm = self.assess_case(
                    case_id=case_data['case_id'],
                    case_text=case_data['case_text'],
                    ground_truth=case_data['reference_markers'],
                    use_memory=True)
                batch_cases.append(cm)
                window.append(cm)
                cases_since_abstraction += 1
                total_cases += 1

                if cases_since_abstraction >= 50:
                    print("\n  🧠 Extracting abstract principles...")
                    self._extract_principles(window[-50:])
                    cases_since_abstraction = 0

                if total_cases % prune_every == 0:
                    rp = self.experience.prune(min_confidence=0.25)
                    sp = self.case_summaries.prune(min_confidence=2.5)
                    if rp+sp > 0:
                        print(f"  ✂️  Pruned {rp} experience rules, {sp} case summaries")

            acc = np.mean([c.performance_metrics['accuracy'] for c in batch_cases])
            marker_accs = {m: np.mean([c.performance_metrics[m] for c in batch_cases])
                           for m in self.markers}
            bm = {'batch_id':batch_idx,'cases_processed':b_end,
                  'batch_accuracy':acc, **marker_accs}
            self.training_metrics.append(bm)

            print(f"\n  Batch {batch_idx+1} results:")
            print(f"    Accuracy                 : {acc:.4f}")
            print(f"    antimicrobials_indicated : {marker_accs['antimicrobials_indicated']:.4f}")
            print(f"    duration_excessive       : {marker_accs['duration_excessive']:.4f}")
            print(f"    incorrect_route          : {marker_accs['incorrect_route']:.4f}")
            print(f"    incorrect_dosing         : {marker_accs['incorrect_dosing']:.4f}")
            yes_rates = self.observational.get_stats()
            print(f"    YES rates: {{k: f'{v:.0%}' for k,v in list(yes_rates.items())[:4]}}")
            print(f"    Experience rules : {len(self.experience.rules)}")
            print(f"    Case summaries   : {len(self.case_summaries.summaries)}")
            print(f"    Principles       : {len(self.abstract.principles)}")

            if checkpoint_mgr and (batch_idx+1) % auto_save_every == 0:
                checkpoint_mgr.save(self, b_end, batch_idx,
                                    self.training_metrics, {'batch_accuracy':acc})

        if checkpoint_mgr:
            checkpoint_mgr.save(self, len(train_cases), batch_idx,
                                 self.training_metrics, {'status':'completed'})
        print(f"\n{'='*70}\nTRAINING COMPLETE\n{'='*70}")
        return self.training_metrics

    def load_from_checkpoint(self, state: Dict):
        m = state['memories']
        self.short_term    = m['short_term']
        self.long_term     = m['long_term']
        self.episodic      = m['episodic']
        self.abstract      = m['abstract']
        self.error_memory  = m['error_memory']
        self.policy        = m['policy']
        self.observational = m['observational']
        self.experience     = m.get('experience',     ExperienceMemory())
        self.case_summaries = m.get('case_summaries', CaseSummaryMemory())
        ctrs = state.get('counters',{})
        self._exp_counter = ctrs.get('exp_counter', len(self.experience.rules))
        self._sum_counter = ctrs.get('sum_counter', len(self.case_summaries.summaries))
        self.training_metrics = state['training_metrics']
        print(f"  ✅ V4b restored | cases:{len(self.long_term.cases)} "
              f"| rules:{len(self.experience.rules)} "
              f"| summaries:{len(self.case_summaries.summaries)}")


# =============================================================================
# EVALUATION (unchanged from V4)
# =============================================================================

def evaluate_only(agent: CAPAgentV4b, test_cases: List[Dict],
                  output_dir: str) -> Dict:
    print(f"\n{'='*70}\nEVALUATION  |  {len(test_cases)} test cases\n{'='*70}\n")
    all_preds = {m:[] for m in agent.markers}
    all_gt    = {m:[] for m in agent.markers}
    all_conf  = {m:[] for m in agent.markers}
    case_rows = []

    for case_data in tqdm(test_cases, desc="Evaluating"):
        cm = agent.assess_case(
            case_id=case_data['case_id'],
            case_text=case_data['case_text'],
            ground_truth=case_data['reference_markers'],
            use_memory=True)
        for m in agent.markers:
            all_preds[m].append(cm.predictions.get(m,'NO'))
            all_gt[m].append(cm.ground_truth.get(m,'NO'))
            all_conf[m].append(cm.confidences.get(m,3))
        case_rows.append({
            'case_id': cm.case_id, 'accuracy': cm.performance_metrics['accuracy'],
            **{f'pred_{m}': cm.predictions.get(m) for m in agent.markers},
            **{f'gt_{m}':   cm.ground_truth.get(m) for m in agent.markers},
        })

    marker_metrics = {}
    for m in agent.markers:
        yt = [1 if x=='YES' else 0 for x in all_gt[m]]
        yp = [1 if x=='YES' else 0 for x in all_preds[m]]
        yc = all_conf[m]
        if len(set(yt)) > 1:
            tn,fp,fn,tp = confusion_matrix(yt,yp,labels=[0,1]).ravel()
        else:
            tn=sum(1 for x in yt if x==0); fp=fn=0; tp=sum(yt)
        prec  = tp/(tp+fp) if (tp+fp)>0 else 0.0
        rec   = tp/(tp+fn) if (tp+fn)>0 else 0.0
        spec  = tn/(tn+fp) if (tn+fp)>0 else 0.0
        npv   = tn/(tn+fn) if (tn+fn)>0 else 0.0
        f1    = f1_score(yt,yp,zero_division=0)
        acc   = float(np.mean(np.array(yt)==np.array(yp)))
        try: auc = roc_auc_score(yt,yc) if len(set(yt))>1 else float('nan')
        except: auc = float('nan')
        marker_metrics[m] = {
            'accuracy':acc,'f1':f1,'recall':rec,'specificity':spec,
            'ppv':prec,'npv':npv,'auc':auc,
            'tp':int(tp),'fp':int(fp),'tn':int(tn),'fn':int(fn)}

    yt_all = [1 if x=='YES' else 0 for m in agent.markers for x in all_gt[m]]
    yp_all = [1 if x=='YES' else 0 for m in agent.markers for x in all_preds[m]]
    yc_all = [c for m in agent.markers for c in all_conf[m]]
    try: auc_all = roc_auc_score(yt_all,yc_all) if len(set(yt_all))>1 else float('nan')
    except: auc_all = float('nan')

    results = {
        'overall_accuracy': float(np.mean(np.array(yt_all)==np.array(yp_all))),
        'overall_f1':       float(f1_score(yt_all,yp_all,zero_division=0)),
        'overall_auc':      float(auc_all),
        'marker_metrics':   marker_metrics,
        'n_test_cases':     len(test_cases),
    }
    os.makedirs(output_dir, exist_ok=True)
    with open(f"{output_dir}/evaluation_results.json",'w') as f:
        json.dump(results,f,indent=2,default=str)
    pd.DataFrame(case_rows).to_csv(f"{output_dir}/detailed_results.csv",index=False)

    print(f"\n  Overall Accuracy : {results['overall_accuracy']:.4f}")
    print(f"  Overall F1       : {results['overall_f1']:.4f}")
    print(f"  Overall AUC      : {results['overall_auc']:.4f}")
    print(f"\n  {'Marker':<30} {'Acc':>6} {'F1':>6} {'Rec':>6} {'Spec':>6} {'AUC':>6}")
    print(f"  {'─'*56}")
    for marker,mm in marker_metrics.items():
        print(f"  {marker:<30} {mm['accuracy']:>6.3f} {mm['f1']:>6.3f} "
              f"{mm['recall']:>6.3f} {mm['specificity']:>6.3f} "
              f"{mm.get('auc',float('nan')):>6.3f}")
    return results


In [ ]:

# ============================================================================
# CELL 5 (DEBUG): EXTRACTION PIPELINE SMOKE TEST
# Run this cell on a single case before training to verify every layer
# is working correctly. Set DEBUG_VERBOSE=True for full output.
# ============================================================================

DEBUG_VERBOSE  = True   # Full output including raw LLM responses
DEBUG_CASE_IDX = 0      # Which case from dataset to use for smoke test

import json, textwrap
from docx import Document

print("=" * 70)
print("V4b DEBUG — EXTRACTION PIPELINE SMOKE TEST")
print("=" * 70)

# ── Load guideline and one case ───────────────────────────────────────────────
doc = Document(GUIDELINE_FILE)
guideline_text = "\n\n".join([p.text.strip() for p in doc.paragraphs if p.text.strip()])
print(f"✅ Guideline loaded: {len(guideline_text):,} chars\n")

all_cases = []
with open(DATA_FILE) as f:
    for line in f:
        all_cases.append(json.loads(line))

case = all_cases[DEBUG_CASE_IDX]
case_id   = case['case_id']
case_text = case['case_text']
gt        = case['reference_markers']

print(f"📋 Case ID    : {case_id}")
print(f"   Text length: {len(case_text)} chars")
print(f"   Ground truth: {gt}")
print()

# ── Init agent (lightweight — reuse if already loaded) ────────────────────────
try:
    agent
    print("♻️  Reusing existing agent in memory\n")
except NameError:
    print("🔧 Initialising fresh agent for debug...\n")
    agent = CAPAgentV4b(guideline_text)

# ─────────────────────────────────────────────────────────────────────────────
# DEBUG LAYER 1: PYTHON PRE-PROCESSING
# ─────────────────────────────────────────────────────────────────────────────
print("─" * 70)
print("LAYER 1: PYTHON PRE-PROCESSING")
print("─" * 70)

precomp = agent._layer1_preprocess(case_text)

# Age / Sex / Weight
print(f"\n  Demographics:")
print(f"    Age    : {precomp.get('age', 'NOT FOUND')}")
print(f"    Sex    : {precomp.get('sex', 'NOT FOUND')}")
print(f"    Weight : {precomp.get('weight_kg', 'NOT FOUND')} kg")

# Renal function
print(f"\n  Renal function:")
print(f"    Creatinine    : {precomp.get('creatinine_umol', 'NOT FOUND')} umol/L")
print(f"    eGFR          : {precomp.get('egfr', 'NOT FOUND')}")
print(f"    CrCl documented: {precomp.get('crcl_documented', 'NOT FOUND')}")
crcl = precomp.get('crcl')
crcl = crcl if (crcl is not None and crcl > 0) else -1
src_label = precomp.get('crcl_source', 'unknown')
if crcl > 0:
    print(f"    CrCl USED     : {crcl:.1f} mL/min  ({src_label})")
    if src_label == 'calculated_CG':
        print(f"    ⚙️  Calculated via Cockcroft-Gault from age/weight/creatinine/sex")
    elif src_label == 'egfr_proxy':
        print(f"    ⚠️  Using eGFR as CrCl proxy (CG not calculable — missing fields)")
    elif src_label == 'documented':
        print(f"    ✅ Directly documented in case")
else:
    print(f"    ⚠️  CrCl could NOT be calculated — insufficient data")
    print(f"       Missing: {[k for k in ['age','sex','weight_kg','creatinine_umol'] if not precomp.get(k)]}")

print(f"    Dialysis      : '{precomp.get('dialysis', 'none')}'")
print(f"    Renal category: {precomp.get('renal_category', 'Unknown')}")

# CORB
print(f"\n  CORB scoring:")
comp = precomp.get('corb_components', {})
for criterion, val in comp.items():
    status = "✅ PRESENT" if val else "❌ absent"
    print(f"    {criterion:<15} : {status}")
print(f"    CORB calculated : {precomp.get('corb_calculated', 'N/A')}")

# Antibiotic history / DOT
print(f"\n  Antibiotic history (regex parsed):")
history = precomp.get('antibiotic_history', [])
if history:
    for h in history:
        print(f"    {h}")
else:
    print(f"    ⚠️  No antibiotic history parsed by regex — relies on LLM extraction")

# IV-oral switch
print(f"\n  IV-to-oral switch criteria (from vitals):")
sw = precomp.get('iv_oral_switch', {})
criteria_keys = ['clinical_stability_gt24h','afebrile_gt24h','heart_rate_lt100',
                  'resp_rate_lt24','sbp_gt90','sats_stable_improving','tolerating_oral']
for k in criteria_keys:
    v = sw.get(k, 'NOT EVALUATED')
    icon = "✅" if v is True else ("❌" if v is False else "❓")
    print(f"    {icon} {k}: {v}")
print(f"    → All met  : {sw.get('all_met', False)}")
print(f"    → How many : {sw.get('how_many', 0)}/7")
if sw.get('not_met'):
    print(f"    → Not met  : {sw.get('not_met', [])}")

# Vitals extracted
print(f"\n  Vitals extracted:")
vitals = precomp.get('vitals', {})
if vitals:
    for k, v in vitals.items():
        print(f"    {k}: {v}")
else:
    print(f"    ⚠️  No vitals extracted from text")

print(f"\n  Layer 1 status: {'✅ OK' if precomp.get('renal_category') != 'Unknown' else '⚠️  Renal category unknown — check age/weight/creatinine in case text'}")

# ─────────────────────────────────────────────────────────────────────────────
# DEBUG LAYER 2: LLM EXTRACTION
# ─────────────────────────────────────────────────────────────────────────────
print()
print("─" * 70)
print("LAYER 2: LLM GUIDELINE-AWARE EXTRACTION")
print("─" * 70)

import time
t0 = time.time()
llm_fields = agent._layer2_llm_extract(case_text, precomp)
elapsed_l2 = time.time() - t0

print(f"\n  ⏱  Extraction time : {elapsed_l2:.2f}s")
print(f"  Output length     : {len(llm_fields)} chars")
print(f"  Lines extracted   : {len([l for l in llm_fields.split(chr(10)) if ':' in l])}")

print(f"\n  ── Raw LLM extraction output ──")
for line in llm_fields.split('\n'):
    line = line.strip()
    if not line: continue
    # Flag missing/not-stated fields
    if 'NOT_STATED' in line or 'not_stated' in line.lower():
        print(f"    ⚠️  {line}")
    elif 'N/A' in line:
        print(f"    ➖  {line}")
    else:
        print(f"    ✅  {line}")

# Check critical fields
print(f"\n  ── Critical field check ──")
critical_fields = [
    'IMAGING_IMPRESSION',
    'CAP_SEVERITY',
    'CORB_SCORE_AT_ONSET',
    'CURRENT_ANTIBIOTIC_NAME',
    'CURRENT_ANTIBIOTIC_DOSE',
    'CURRENT_ANTIBIOTIC_ROUTE',
    'PRIOR_ANTIBIOTICS',
    'CURRENT_THERAPY_DAY',
    'ALLERGIES_AND_REACTIONS',
    'MICROBIOLOGY',
    'SWITCH_CRITERIA_1',
    'AGE_YEARS',
    'WEIGHT_KG',
    'RENAL_FUNCTION_DOCUMENTED',
]
llm_lower = llm_fields.lower()
missing_critical = []
for field in critical_fields:
    present = field.lower() in llm_lower
    has_value = present and 'not_stated' not in llm_lower.split(field.lower())[-1][:50]
    icon = "✅" if has_value else ("⚠️ " if present else "❌")
    if not present: missing_critical.append(field)
    print(f"    {icon} {field}")

if missing_critical:
    print(f"\n  ⛔ MISSING FIELDS (not extracted at all): {missing_critical}")
    print(f"     → These fields will not generate GUIDELINE_FLAGS")

# ─────────────────────────────────────────────────────────────────────────────
# DEBUG LAYER 3: ANNOTATION
# ─────────────────────────────────────────────────────────────────────────────
print()
print("─" * 70)
print("LAYER 3: PYTHON GUIDELINE FLAG ANNOTATION")
print("─" * 70)

t0 = time.time()
feature_summary = agent._layer3_annotate(llm_fields, precomp, case_text)
elapsed_l3 = time.time() - t0

print(f"\n  ⏱  Annotation time : {elapsed_l3:.4f}s (pure Python)")

# Extract just the GUIDELINE_CONTEXT block
if 'GUIDELINE_CONTEXT:' in feature_summary:
    flag_block = feature_summary.split('GUIDELINE_CONTEXT:')[-1].strip()
    flag_lines = [l.strip() for l in flag_block.split('\n') if l.strip()]
    print(f"  Flags generated  : {len(flag_lines)}")
    print(f"\n  ── Guideline flags ──")
    for line in flag_lines:
        if 'likely YES' in line or '⚠️' in line or 'MISMATCH' in line:
            print(f"    🚨 {line}")
        elif 'is NO' in line or 'appropriate' in line.lower():
            print(f"    ✅ {line}")
        elif 'RENAL_CATEGORY' in line:
            print(f"    🔬 {line}")
        elif 'EFFECTIVE_DOT' in line:
            print(f"    📅 {line}")
        else:
            print(f"    ℹ️  {line}")
else:
    print("  ⛔ GUIDELINE_CONTEXT block NOT FOUND in feature summary")
    print("     → Layer 3 annotation may have failed silently")

# Total feature summary stats
print(f"\n  ── Feature summary stats ──")
print(f"    Total chars    : {len(feature_summary)}")
print(f"    Total lines    : {len(feature_summary.split(chr(10)))}")
print(f"    Embedding chars: {min(len(feature_summary), 512)} (first 512 used for embed)")

# ─────────────────────────────────────────────────────────────────────────────
# DEBUG DOSING TABLE LOOKUP
# ─────────────────────────────────────────────────────────────────────────────
print()
print("─" * 70)
print("DOSING TABLE LOOKUP DEBUG")
print("─" * 70)

import re
fs_lower = feature_summary.lower()
drug_m = re.search(r'current_antibiotic_name[:\s]*([^\n\[]+)', fs_lower)
route_m = re.search(r'current_antibiotic_route[:\s]*([^\n\[]+)', fs_lower)
drug_raw  = drug_m.group(1).strip() if drug_m else ''
route_raw = route_m.group(1).strip() if route_m else ''

print(f"\n  Raw drug extracted  : '{drug_raw}'")
print(f"  Raw route extracted : '{route_raw}'")

drug_canonical = _normalise_drug(drug_raw) if drug_raw else ''
print(f"  Canonical name      : '{drug_canonical}'")

renal_cat = precomp.get('renal_category', 'Unknown')
print(f"  Renal category      : {renal_cat}")

recommended = _lookup_dose(drug_canonical, renal_cat) if drug_canonical else None
if recommended:
    print(f"\n  ✅ Dosing table lookup SUCCESS")
    print(f"     Recommended: {recommended}")
    dose_m = re.search(r'current_antibiotic_dose[^\n]*[:\s]*([^\n]+)', fs_lower)
    prescribed = dose_m.group(1).strip() if dose_m else 'NOT EXTRACTED'
    print(f"     Prescribed : {prescribed}")
    in_no_renal = drug_canonical in NO_RENAL_ADJUSTMENT
    print(f"     Renal adjustment needed: {'NO (not renally cleared)' if in_no_renal else 'YES (renally cleared)'}")
else:
    print(f"\n  ⚠️  No dosing table entry for '{drug_canonical}' + '{renal_cat}'")
    print(f"     → incorrect_dosing decision left entirely to LLM")
    # Try to diagnose why
    if not drug_canonical:
        print(f"     Cause: Drug name not extracted from LLM output")
    elif renal_cat == 'Unknown':
        print(f"     Cause: Renal category unknown — cannot look up dose")
    else:
        similar = [k for k in DOSING_TABLE.keys() if drug_canonical[:6] in k[0]]
        if similar:
            print(f"     Similar keys in table: {similar[:3]}")
        else:
            print(f"     Drug not in DOSING_TABLE at all — may need DRUG_ALIASES entry")

# ─────────────────────────────────────────────────────────────────────────────
# DEBUG SPECTRUM CHECK
# ─────────────────────────────────────────────────────────────────────────────
print()
print("─" * 70)
print("SPECTRUM CHECK DEBUG")
print("─" * 70)

sev_m = re.search(r'cap_severity[:\s]*([^\n\[]+)', fs_lower)
severity_raw = sev_m.group(1).strip() if sev_m else 'unknown'
severity = 'moderate'
for s in ['mild','moderate','severe']:
    if s in severity_raw: severity = s; break

print(f"\n  CAP severity (extracted): '{severity_raw}' → using '{severity}'")
print(f"  Drug for spectrum check : '{drug_raw}'")

if drug_raw:
    too_broad  = _drug_too_broad(drug_raw, severity)
    too_narrow = _drug_too_narrow(drug_raw, severity)
    if too_broad:
        print(f"  🚨 SPECTRUM: {drug_raw} is TOO BROAD for {severity} CAP")
    elif too_narrow:
        print(f"  🚨 SPECTRUM: {drug_raw} is TOO NARROW for {severity} CAP")
    else:
        print(f"  ✅ SPECTRUM: {drug_raw} appears appropriate for {severity} CAP")
else:
    print(f"  ⚠️  No drug extracted — spectrum check skipped")

# ─────────────────────────────────────────────────────────────────────────────
# DEBUG ONE DECISION CALL (antimicrobials_indicated)
# ─────────────────────────────────────────────────────────────────────────────
print()
print("─" * 70)
print("SAMPLE DECISION CALL: antimicrobials_indicated")
print("─" * 70)

feature_embedding = agent.embed(feature_summary)
print(f"\n  Embedding shape: {feature_embedding.shape}")

marker = 'antimicrobials_indicated'
prompt, ret_rules, ret_sums = agent._build_decision_prompt(
    feature_summary, marker, feature_embedding, use_memory=False)

print(f"  Decision prompt length : {len(prompt)} chars")
print(f"  Experience rules retrieved : {len(ret_rules)}")
print(f"  Case summaries retrieved   : {len(ret_sums)}")

if DEBUG_VERBOSE:
    print(f"\n  ── Decision prompt (first 600 chars) ──")
    print(textwrap.indent(prompt[:600], '    '))
    print(f"    ... [truncated]")

t0 = time.time()
response = agent._generate(prompt, max_new_tokens=30)
elapsed_dec = time.time() - t0

print(f"\n  ⏱  Decision time : {elapsed_dec:.2f}s")
print(f"  Raw LLM response : '{response[-100:]}'")  # last 100 chars

pred, conf = agent._parse_decision(response)
print(f"  Parsed  → PREDICTION={pred}  CONFIDENCE={conf}")

pred_after_sanity = agent._sanity_check(
    marker, pred, feature_summary, case_text, precomp)
print(f"  After sanity check → {pred_after_sanity}")
print(f"  Ground truth       → {gt.get(marker, 'UNKNOWN')}")
correct = pred_after_sanity == gt.get(marker)
print(f"  Correct            → {'✅ YES' if correct else '❌ NO'}")

# ─────────────────────────────────────────────────────────────────────────────
# TIMING SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print()
print("─" * 70)
print("TIMING SUMMARY (single case estimate)")
print("─" * 70)
print(f"\n  Layer 1 (Python pre-process) : ~0.001s  (pure Python)")
print(f"  Layer 2 (LLM extraction)     : {elapsed_l2:.2f}s")
print(f"  Layer 3 (Python annotation)  : {elapsed_l3:.4f}s  (pure Python)")
print(f"  1 decision call              : {elapsed_dec:.2f}s")
print(f"  Estimated 8 decision calls   : {elapsed_dec*8:.2f}s")
est_reflexion = elapsed_dec * 1.5
est_summary   = elapsed_dec * 1.2
print(f"  Estimated batch reflexion    : ~{est_reflexion:.2f}s (conditional)")
print(f"  Estimated 3 case summaries   : ~{est_summary*3:.2f}s (conditional)")
est_total = 0.001 + elapsed_l2 + elapsed_l3 + (elapsed_dec*8) + est_reflexion + (est_summary*3)
print(f"\n  ── Estimated total per case  : {est_total:.1f}s ──")
print(f"  ── Estimated per 50-case batch: {est_total*50/60:.1f} min ──")
print(f"\n  ⚠️  Reflexion + summary calls are conditional:")
print(f"     Fewer errors over time → less reflexion → faster batches")

print()
print("=" * 70)
print("SMOKE TEST COMPLETE — check warnings above before running training")
print("=" * 70)


In [ ]:
# ============================================================================
# CELL 5: TRAINING SESSION
# ============================================================================
from docx import Document
import json, os

print("=" * 70)
print("V4b TRAINING SESSION")
print("=" * 70)

doc = Document(GUIDELINE_FILE)
guideline_text = "\n\n".join([p.text.strip() for p in doc.paragraphs if p.text.strip()])
print(f"✅ Guideline loaded: {len(guideline_text):,} chars")

all_cases = []
with open(DATA_FILE) as f:
    for line in f:
        all_cases.append(json.loads(line))

NUM_TRAIN   = 1000
train_cases = all_cases[:NUM_TRAIN]
print(f"✅ {len(train_cases)} train  |  {len(all_cases)-NUM_TRAIN} test")

checkpoint_mgr = CheckpointManager(paths.CHECKPOINT_DIR)
start_batch = 0

if RESUME_FROM:
    print(f"\n🔄 Resuming from: {RESUME_FROM}")
    state = CheckpointManager(os.path.dirname(RESUME_FROM)).load_checkpoint(RESUME_FROM)
    if state:
        agent = CAPAgentV4b(guideline_text)
        agent.load_from_checkpoint(state)
        start_batch = state['batch_id'] + 1
    else:
        print("❌ Checkpoint not loaded — starting fresh")
        agent = CAPAgentV4b(guideline_text)
else:
    agent = CAPAgentV4b(guideline_text)

metrics = agent.train(
    train_cases      = train_cases,
    batch_size       = 50,
    checkpoint_mgr   = checkpoint_mgr,
    start_from_batch = start_batch,
    max_batches      = 20,
    auto_save_every  = 1,
    prune_every      = 200,
)

print("\n✅ Training complete!")
print(f"   Experience rules : {len(agent.experience.rules)}")
print(f"   Case summaries   : {len(agent.case_summaries.summaries)}")
print(f"   Principles       : {len(agent.abstract.principles)}")


## 💾 Saving Checkpoints Between Sessions

1. **Save version** (top right) → do NOT tick "Save & Run All"
2. **Output tab** → find `/kaggle/working/checkpoints/` → **… → Add to dataset**
3. Name it (e.g. `cap-v4b-batch-10`) and save
4. Next session: **Add data** → attach that dataset → set `RESUME_FROM` in Cell 3

> V4b checkpoints include all 9 memory systems.
> V4b can also resume from V4 checkpoints — new dosing/spectrum logic is Python-only
> and requires no stored state.


In [ ]:
# ============================================================================
# CELL 6: EVALUATION
# ============================================================================
from docx import Document
import json, os

print("=" * 70)
print("V4b EVALUATION SESSION")
print("=" * 70)

doc = Document(GUIDELINE_FILE)
guideline_text = "\n\n".join([p.text.strip() for p in doc.paragraphs if p.text.strip()])

try:
    agent
    print("✅ Using agent already in memory")
except NameError:
    cp_path = RESUME_FROM or f"{paths.CHECKPOINT_DIR}/latest_checkpoint.pkl"
    print(f"📦 Loading checkpoint: {cp_path}")
    state = CheckpointManager(os.path.dirname(cp_path)).load_checkpoint(cp_path)
    if not state:
        raise RuntimeError("No checkpoint found. Run training first.")
    agent = CAPAgentV4b(guideline_text)
    agent.load_from_checkpoint(state)

all_cases = []
with open(DATA_FILE) as f:
    for line in f:
        all_cases.append(json.loads(line))

test_cases = all_cases[1000:]
print(f"🧪 {len(test_cases)} test cases")
results = evaluate_only(agent, test_cases, paths.EVAL_DIR)
print(f"\n✅ Results saved → {paths.EVAL_DIR}")


In [ ]:
# ============================================================================
# CELL 7: VIEW & COMPARE RESULTS
# ============================================================================
import json, pandas as pd, numpy as np

with open(f"{paths.EVAL_DIR}/evaluation_results.json") as f:
    results = json.load(f)

print("=" * 70)
print("EVALUATION RESULTS — V4b")
print("=" * 70)
print(f"  Overall Accuracy : {results['overall_accuracy']:.4f}")
print(f"  Overall F1       : {results['overall_f1']:.4f}")
print(f"  Overall AUC      : {results.get('overall_auc', 'n/a')}")

print(f"\n  {'Marker':<30} {'Acc':>6} {'F1':>6} {'Rec':>6} {'Spec':>6} {'PPV':>6} {'NPV':>6} {'AUC':>6}")
print(f"  {'─'*76}")
for marker, m in results['marker_metrics'].items():
    print(f"  {marker:<30} "
          f"{m['accuracy']:>6.3f} {m['f1']:>6.3f} "
          f"{m['recall']:>6.3f} {m['specificity']:>6.3f} "
          f"{m['ppv']:>6.3f} {m['npv']:>6.3f} "
          f"{m.get('auc', float('nan')):>6.3f}")

print("\n" + "=" * 70)
print("BASELINE (V1) vs V4b")
print("=" * 70)
baseline = {
    'overall_accuracy': 0.789, 'overall_f1': 0.461,
    'antimicrobials_indicated': 0.381, 'duration_excessive': 0.746,
    'allergy_mismatch': 0.847, 'incorrect_route': 0.814,
}
rows = []
for key, bv in baseline.items():
    if key.startswith('overall_'):
        metric = key.replace('overall_', '')
        ev = results.get(f'overall_{metric}', float('nan'))
    else:
        ev = results['marker_metrics'].get(key, {}).get('accuracy', float('nan'))
    diff = ev - bv
    rows.append({'Metric': key, 'Baseline(V1)': f"{bv:.4f}", 'V4b': f"{ev:.4f}",
                 'Change': f"{diff:+.4f}", 'Change%': f"{diff/bv*100:+.1f}%"})
print(pd.DataFrame(rows).to_string(index=False))

detailed = pd.read_csv(f"{paths.EVAL_DIR}/detailed_results.csv")
print(f"\nTest cases   : {len(detailed)}")
print(f"Mean accuracy: {detailed['accuracy'].mean():.4f}")
print(f"Perfect (8/8): {(detailed['accuracy']==1.0).sum()}")
print(f"≥7/8 correct : {(detailed['accuracy']>=0.875).sum()}")


In [ ]:
# ============================================================================
# CELL 8: MONITOR MEMORY STATE
# ============================================================================
import pickle
from pathlib import Path

checkpoint_dir = Path(paths.CHECKPOINT_DIR)
checkpoints    = sorted(checkpoint_dir.glob("checkpoint_batch_*.pkl"))

if not checkpoints:
    print("No checkpoints found yet.")
else:
    print(f"{'='*70}")
    print(f"Checkpoints found: {len(checkpoints)}")
    print(f"{'='*70}")
    for cp in checkpoints:
        with open(cp,'rb') as f: d=pickle.load(f)
        last = d['training_metrics'][-1] if d['training_metrics'] else {}
        print(f"  {cp.name}  cases={d['cases_processed']}  "
              f"acc={last.get('batch_accuracy',0):.3f}  {d['timestamp'][:19]}")

    with open(checkpoints[-1],'rb') as f: d=pickle.load(f)
    m = d['memories']
    print(f"\n  Memory state (latest):")
    print(f"    Long-term cases   : {len(m['long_term'].cases)}")
    print(f"    Error reflexions  : {len(m['error_memory'].errors)}")
    print(f"    Principles        : {len(m['abstract'].principles)}")

    exp = m.get('experience')
    sums = m.get('case_summaries')
    if exp:
        print(f"    Experience rules  : {len(exp.rules)}")
        top = sorted(exp.rules, key=lambda r: r.confidence, reverse=True)[:5]
        if top:
            print(f"\n  Top experience rules by confidence:")
            for r in top:
                print(f"    [{r.confidence:.0%}] [{r.marker}] {r.clinical_rule[:90]}")
    if sums:
        print(f"    Case summaries    : {len(sums.summaries)}")
        top = sorted(sums.summaries, key=lambda s: s.times_confirmed, reverse=True)[:3]
        if top:
            print(f"\n  Most-confirmed case summaries:")
            for s in top:
                print(f"    [confirmed {s.times_confirmed}x] [{s.marker}={s.prediction}] {s.case_pattern[:90]}")

    if m['abstract'].principles:
        print(f"\n  Sample abstract principles:")
        for pid,p in list(m['abstract'].principles.items())[:3]:
            print(f"    [{p.confidence:.0%}] {p.description[:100]}")


In [ ]:
# ============================================================================
# CELL 9: SAVE OUTPUTS
# ============================================================================
import os
from pathlib import Path

print("Files that will be saved on commit:\n")
for folder in [paths.CHECKPOINT_DIR, paths.EVAL_DIR, paths.OUTPUT_DIR]:
    files = [f for f in Path(folder).rglob("*") if f.is_file()]
    if files:
        print(f"📁 {folder}/")
        for f in files:
            print(f"   {f.name:<45} {f.stat().st_size/1024:>8.1f} KB")
        print()
print("✅ On  Save version → Commit  all these files will be preserved.")
